# Snowflake

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/snowflake.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Snowflake is a fully managed, multi-cluster, shared-data cloud data platform: storage, compute, and services scale independently. This notebook walks from the architecture that makes that true through loading, transformation, governance, and cost, then finishes with a tuning chapter that turns the system functions and account-usage views into a step-by-step performance workflow.

## What is Snowflake?

Snowflake is a fully managed cloud data platform delivered as a service. You do not provision servers, install a database, choose a distribution key, or run `ANALYZE`: you create an account, load data, and write SQL. The mental model to hold is a shared-data architecture, where one governed copy of the data lives in cloud object storage and separate compute clusters all read it.

The platform has three layers. Storage holds table data as compressed, columnar micro-partitions in object storage. Compute is made of virtual warehouses, independent clusters that execute queries and DML. Cloud services is the control plane that authenticates users, parses and optimizes SQL, keeps metadata and transactions, and provisions warehouses. Each layer scales and is billed on its own.

That separation is why Snowflake behaves differently from a traditional warehouse. There is no cluster to expand, no vacuum to schedule, and no index to choose. Adding a warehouse gives a new team isolated compute over the same tables with zero copies, and resizing a warehouse changes speed without touching data. ACID transactions, semi-structured data, Time Travel, and cross-account sharing are built in rather than bolted on.

The gotcha is that fully managed does not mean automatically fast or automatically cheap. Compute is billed per second of warehouse runtime, so an always-on warehouse costs money while doing nothing, and a large warehouse cannot rescue a query that scans every micro-partition. The levers you actually own are warehouse size and lifecycle, table ordering and clustering, query shape, and caching.

```text
clients (Snowsight, JDBC/ODBC, Python, dbt, Spark)
   -> cloud services (metadata, optimizer, transactions, security, governance)
   -> virtual warehouses (isolated compute) over storage (micro-partitions in S3/Blob/GCS)
```

```sql
-- Confirm where you are connected and what compute exists.
SELECT CURRENT_ACCOUNT()  AS account,
       CURRENT_REGION()   AS region,
       CURRENT_VERSION()  AS version,
       CURRENT_ROLE()     AS role,
       CURRENT_WAREHOUSE() AS warehouse;

SHOW WAREHOUSES;

SELECT table_catalog, table_schema, COUNT(*) AS tables
FROM analytics.information_schema.tables
GROUP BY 1, 2
ORDER BY tables DESC;
```

Snowflake separates storage, compute, and services so each scales and is billed independently, and your job shifts from tuning servers to shaping queries and sizing warehouses.

The three layers are billed differently. Storage is charged per terabyte per month. A warehouse is charged in credits per second of uptime. Cloud services is charged only for the part of a day's cloud-services consumption that exceeds 10 percent of that day's warehouse compute credits, so ordinary parsing, optimization, and metadata work is effectively included. Serverless features sit outside that allowance: Snowpipe, automatic clustering, and materialized-view maintenance consume Snowflake-managed compute and are billed as serverless credits attributed to the feature that used them.

A concrete sense of scale: a MEDIUM warehouse running 8 hours a day for 22 working days consumes 4 x 8 x 22 = 704 credits. Running the same workload on XSMALL takes roughly four times as long but consumes nearly the same credits, because the credit rate and the runtime move in opposite directions. The two variables that actually change the bill are total warehouse uptime and how many bytes each statement reads.

The subtle trap is that the 10 percent cloud-services allowance is a daily, account-wide calculation. A day with very little warehouse compute makes the allowance tiny, so a metadata-heavy day can show billed cloud-services credits even though the workload never started a warehouse.

## How does Snowflake separate storage and compute?

Storage is a single logical data store backed by cloud object storage such as S3, Azure Blob Storage, or Google Cloud Storage. When you load rows, Snowflake writes them into micro-partitions, encrypts them, and keeps them there. You never address a file or a node directly; you query a table and the service decides which partitions to read. Storage is billed per terabyte per month, including Time Travel and Fail-safe copies.

Compute is a fleet of virtual warehouses. Each warehouse is its own MPP cluster with its own CPU, memory, and local SSD cache, and warehouses share no compute resources with each other. They all read the same tables from storage. Two warehouses running at once therefore cannot slow each other down, which is what people mean when they say Snowflake isolates workloads.

Because compute is elastic, workload isolation becomes configuration rather than infrastructure. A small BI warehouse, a large ELT warehouse, and an ad-hoc warehouse for analysts can all exist at once, and you create or drop them in seconds. The tradeoff is that each warehouse warms its own cache, so a query that runs on a different warehouse pays to read from storage again even if another warehouse just read the same rows.

The practical gotcha: separation does not reduce the bytes a query must read. If a filter cannot prune micro-partitions, every partition is still scanned, just faster on a bigger warehouse. Fix pruning, filtering, and query shape first; use compute size to shorten the work that genuinely remains.

```sql
-- Give each workload its own compute with its own lifecycle.
CREATE WAREHOUSE bi_wh        WAREHOUSE_SIZE = 'SMALL'  AUTO_SUSPEND = 300 AUTO_RESUME = TRUE;
CREATE WAREHOUSE elt_wh       WAREHOUSE_SIZE = 'LARGE'  AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;
CREATE WAREHOUSE adhoc_wh     WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;

-- Compute can be resized without touching data.
ALTER WAREHOUSE elt_wh SET WAREHOUSE_SIZE = 'XLARGE';
ALTER WAREHOUSE elt_wh SET WAREHOUSE_SIZE = 'LARGE';

-- Warehouse uptime and credit burn are visible per warehouse.
SELECT warehouse_name, warehouse_size, state, auto_suspend, auto_resume
FROM TABLE(information_schema.warehouse_load_history(
  date_range_start => DATEADD('hour', -3, CURRENT_TIMESTAMP())));
```

Storage and compute are decoupled services, so isolation and elasticity are configuration — but a bigger warehouse only helps when the work itself can be split.

The layers coordinate through cloud services rather than through each other. Storage holds many versions of each micro-partition, and the transaction manager records which version belongs to which committed transaction. A warehouse receives a plan that already names the partition versions to read, so two warehouses never need to agree on a cache state or a lock. A commit becomes visible to other warehouses as soon as the transaction manager publishes the new version.

```sql
-- Same query text, same unchanged tables, a different warehouse.
USE WAREHOUSE bi_wh;
SELECT COUNT(*) FROM analytics.orders;

USE WAREHOUSE adhoc_wh;
SELECT COUNT(*) FROM analytics.orders;   -- served by the account-wide result cache

-- A new commit invalidates that result and the next run reads storage again.
INSERT INTO analytics.orders (order_id, order_date, amount)
VALUES (999999, '2026-10-04', 10.00);
```

Isolation is a compute property, not a cache property. An identical query can be answered for a different warehouse from the shared result cache, so a two-warehouse benchmark can accidentally measure the cache instead of the new cluster; run the first query on each warehouse with different text or a changed table.

## What are virtual warehouses and how do they work?

A virtual warehouse is a named cluster of compute nodes that executes queries and DML. Sizes run from XSMALL to 6XLARGE, and each step doubles the number of nodes, roughly doubling both throughput and the credit rate per hour. Snowflake bills per second of running time with a 60-second minimum each time a warehouse starts, so many short-lived warehouses can cost more than one warm warehouse.

Inside a warehouse, a query is broken into stages and parallelized across nodes and worker threads. Scans, joins, sorts, and aggregations all benefit from more nodes, which is why doubling a warehouse roughly halves the runtime of a large, parallelizable query while consuming about the same number of credits overall. The wins taper off when one thread or one partition dominates, for example a single-row lookup or a query with a skew-heavy join key.

Warehouses run more than SELECT statements. `COPY INTO` loads, `INSERT`, `UPDATE`, `DELETE`, and `MERGE` all execute on a warehouse. Serverless features such as tasks, Snowpipe, automatic clustering, materialized-view maintenance, and query acceleration use Snowflake-managed compute instead and appear as serverless credits rather than warehouse credits.

The gotcha is the idle warehouse. A warehouse that is `STARTED` bills even when no query is running, which is what `AUTO_SUSPEND` exists to prevent. Also remember that a warehouse holds no table data: it holds a local cache that disappears when the warehouse is suspended, so aggressive auto-suspend saves credits but costs a warm cache on the next query.

```sql
CREATE OR REPLACE WAREHOUSE transform_wh
  WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';   -- heavy backfill
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'MEDIUM';  -- back to steady state

-- See current state and size for every warehouse in the account.
SHOW WAREHOUSES;

-- Suspend explicitly at the end of a batch window.
ALTER WAREHOUSE transform_wh SUSPEND;
```

```text
XS   1 credit/hour      dashboards, dev, small transforms
S    2 credits/hour     light BI, moderate dbt models
M    4 credits/hour     standard ELT, mixed concurrency
L    8 credits/hour     large joins, big backfills
XL  16 credits/hour     heavy transformations
2XL 32 credits/hour     rare; usually a signal to fix the query
```

A virtual warehouse is a metered cluster you start, size, and stop; isolate it by workload, never leave it idle, and remember that it changes speed, not the amount of data a query must read.

A warehouse is a pool of worker threads, not just a size. When more statements arrive than the pool can run at once, the extra statements wait in a queue, and `QUEUED_OVERLOAD_TIME` records that wait separately from `EXECUTION_TIME`. Queueing is a concurrency symptom, not a SQL symptom: the same query that completes in two seconds alone can sit in a queue for thirty seconds behind other work.

```text
XSMALL     1 credit/hour
SMALL      2
MEDIUM     4
LARGE      8
XLARGE    16
2XLARGE   32
3XLARGE   64
4XLARGE  128
5XLARGE  256
6XLARGE  512
```

In a multi-cluster warehouse the rate applies per cluster, so a warehouse allowed to reach six clusters can burn six times the listed rate. The size is only half the cost story; the cluster count and the suspend behavior are the other half.

## What is the cloud services layer?

Cloud services is the control plane that Snowflake operates for you. It handles authentication and access control, session and transaction management, SQL parsing and optimization, metadata storage, and warehouse provisioning. When you submit a query, it is cloud services that compiles it, consults micro-partition statistics to decide what to prune, and hands an execution plan to a warehouse.

The metadata it stores is the reason many Snowflake operations are instant. Table definitions, column statistics, partition ranges, object versions for Time Travel, and privilege grants all live in cloud services. `SHOW` commands, `INFORMATION_SCHEMA`, and `ACCOUNT_USAGE` views read from this layer, which is why some account-usage views lag by up to about 45 minutes to two hours while `INFORMATION_SCHEMA` views are close to real time.

Most cloud-services work is included at no additional charge; the credit columns exist because serverless features are metered. Snowpipe, automatic clustering, materialized-view maintenance, search optimization maintenance, and the query acceleration service consume compute that Snowflake manages, and each query records any cloud-services credits it consumed in `CREDITS_USED_CLOUD_SERVICES`.

The gotcha is metadata-heavy workloads. A pipeline that issues hundreds of thousands of tiny DDL or metadata operations, or a design that runs enormous numbers of very small queries, spends disproportionate time in cloud services and can show up in billing even when warehouses look idle. Batch small operations and prefer set-based SQL over row-by-row procedural loops.

```sql
SELECT query_id, total_elapsed_time / 1000 AS seconds,
       compilation_time / 1000 AS compile_s,
       credits_used_cloud_services,
       partitions_scanned, partitions_total,
       bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -24, CURRENT_TIMESTAMP())
ORDER BY total_elapsed_time DESC
LIMIT 20;

-- Serverless credits are attributed to the feature that consumed them.
SELECT service_type, SUM(credits_used) AS credits
FROM snowflake.account_usage.metering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY service_type
ORDER BY credits DESC;
```

Cloud services is Snowflake's always-on brain: it compiles queries, holds the metadata that enables pruning, and is free for ordinary work but metered for the serverless features it runs.

The 10 percent rule is the detail worth knowing precisely. Snowflake adds up the cloud-services credits and the warehouse compute credits for each day and bills cloud services only when they exceed 10 percent of that day's warehouse compute. The `ACCOUNT_USAGE` views show all consumed cloud-services credits, while `METERING_DAILY_HISTORY` shows what was actually billed, which is why the two numbers do not agree.

Cloud services also serves the result cache. A cache hit returns a stored result in milliseconds without starting a warehouse, and a cache miss is compiled into a plan and dispatched to compute. Because compilation and optimization happen here, a very complex query can spend noticeable time in `COMPILATION_TIME` before any warehouse work begins.

Do not add `CREDITS_USED_CLOUD_SERVICES` from `QUERY_HISTORY` straight onto the bill. Most of that consumption is absorbed by the 10 percent allowance, and only the daily excess is charged. Remember too that `ACCOUNT_USAGE` views can lag behind real time, so a query run a minute ago may not appear yet.

## What are micro-partitions and how does pruning work?

A micro-partition is the unit of storage in a Snowflake table: an immutable, compressed, columnar group of rows that holds roughly 50 to 500 MB of uncompressed data. Tables are partitioned automatically as rows are inserted, with no DDL and no manual maintenance. Because partitions are immutable, a row update or delete records a new partition version rather than rewriting the table.

For every column in every micro-partition, Snowflake stores metadata: the minimum and maximum value, the number of distinct values, and the null count. When a query has a predicate, the optimizer uses those ranges to decide which partitions can contain a matching row and reads only those. That skipping is called pruning, and it is the main reason a Snowflake query over a huge table can still be fast.

Pruning depends on how rows are ordered at write time. A plain `INSERT` places rows in the order they arrive, so a table loaded from unordered files ends up with every date range spread across every partition. Sort the data as you load it (`ORDER BY` in the insert, or ordered source files) and each partition covers a narrow range, which is exactly the layout pruning needs. For very large tables, optionally declare a clustering key to maintain that order as data changes.

The gotcha is that anything that hides the literal value from the optimizer defeats pruning. Wrapping the filter column in a function or cast, comparing a string column to a number with implicit conversion, or joining on mismatched types all prevent the min/max ranges from being used, and the query scans the table instead. Keep predicates on the raw column and compare like with like.

```sql
-- Ordered load: rows arrive sorted, so partition ranges are narrow.
INSERT INTO analytics.orders (order_id, order_date, customer_id, amount)
SELECT order_id, order_date, customer_id, amount
FROM staging.orders_raw
ORDER BY order_date, customer_id;

-- Good predicate: a literal range prunes partitions.
SELECT COUNT(*) FROM analytics.orders
WHERE order_date BETWEEN '2026-09-01' AND '2026-09-30';

-- Bad predicate: the cast hides the literal from partition pruning.
SELECT COUNT(*) FROM analytics.orders
WHERE CAST(order_date AS STRING) LIKE '2026-09%';

-- Measure pruning after the fact: scanned versus total partitions.
SELECT ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       partitions_scanned, partitions_total
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE '%analytics.orders%'
ORDER BY start_time DESC
LIMIT 5;
```

Micro-partitions plus per-column min/max statistics are the whole performance model: order rows by your common filter and pruning does the rest, while functions and type mismatches on the filter column throw it away.

A partition is closed when it reaches the target size, so the order of an insert becomes the physical order of the table. Snowflake does not reshuffle rows to fit a boundary and never rewrites a partition in place; an update or delete records the new row image in a new partition and marks the old one superseded. Statistics are recorded per column per partition, which is what makes a predicate on any single column eligible for pruning.

```text
unordered load                     ordered load
P1  2026-01-01 .. 2026-12-31       P1  2026-01-01 .. 2026-01-07
P2  2026-01-01 .. 2026-12-31       P2  2026-01-08 .. 2026-01-14
P3  2026-01-01 .. 2026-12-31       P3  2026-01-15 .. 2026-01-21
P4  2026-01-01 .. 2026-12-31       P4  2026-01-22 .. 2026-01-28

one-day filter -> 4 partitions      one-day filter -> 1 partition
```

An update breaks the ordering that a sorted load created. The rewritten rows land in new partitions that may span the whole date range again, so a table with continuous updates drifts back toward full scans unless it is reclustered or its load order is restored.

## What is a clustering key and when should you use one?

A clustering key tells Snowflake to keep a large table physically ordered by one or more expressions. It is not an index and not a sort key in the traditional sense: Snowflake stores rows in micro-partitions whose min/max ranges overlap, and clustering tries to reduce that overlap so a range predicate can skip more partitions. The measure of success is clustering depth, the average number of partitions that overlap a point in the clustering space.

Automatic clustering runs as a serverless background process after DML and reorganizes partitions to restore the requested order. You do not schedule it; Snowflake decides when the accumulated disorder justifies the work. That means clustering improves pruning only gradually on a busy table and consumes serverless credits continuously while it catches up.

Use a clustering key only when all of the following are true: the table is large (typically multiple terabytes or a very high row count), workloads repeatedly filter or join on the same column set, `SYSTEM$CLUSTERING_INFORMATION` shows high depth or poor pruning, and the queries are important enough to justify maintenance cost. A good key usually starts with a low-cardinality or time column and ends with a higher-cardinality identifier, for example `(order_date, customer_id)`, because range filters on the leading column benefit most.

The gotcha is over-clustering. Keys on small tables, on columns nobody filters, or low-selectivity columns buy nothing and burn credits on maintenance. Clustering also does not help point lookups on a unique key; that is the search optimization service's job. Always measure depth and query improvement before and after, and suspend clustering when a table stops changing or the key stops paying.

```sql
-- Declare the key: date first for range filters, then a high-cardinality id.
ALTER TABLE analytics.events CLUSTER BY (event_date, customer_id);

-- Depth: lower is better, and around 1 means near-perfect ordering.
SELECT SYSTEM$CLUSTERING_INFORMATION('analytics.events', '(event_date, customer_id)');

-- Depth without passing the key echoes the table's current clustering.
SELECT SYSTEM$CLUSTERING_INFORMATION('analytics.events');

-- Pause maintenance on a table that has stopped changing.
ALTER TABLE analytics.events SUSPEND RECLUSTERING;
-- Resume, or remove the key entirely.
ALTER TABLE analytics.events RESUME RECLUSTERING;
ALTER TABLE analytics.events DROP CLUSTERING KEY;

-- Serverless credit cost of automatic clustering, attributed per table.
SELECT table_name, SUM(credits_used) AS clustering_credits
FROM snowflake.account_usage.automatic_clustering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY table_name
ORDER BY clustering_credits DESC;
```

Cluster only large, repeatedly filtered tables, lead the key with the column used in range filters, and judge it by clustering depth and query improvement against the serverless credits it consumes.

Automatic clustering compares the table's current depth against the requested key and rewrites overlapping partitions until the order is restored. It does not run on a fixed schedule; Snowflake triggers it as disorder accumulates, so cost arrives in bursts after large loads. `SYSTEM$CLUSTERING_INFORMATION` reports `average_depth` and `average_overlaps`, where depth is the average number of partitions that must be probed for a point in the key space.

```text
key (order_date, customer_id)
before reclustering:  average_depth = 3.4   -> a point lookup probes about 3 partitions
after reclustering:   average_depth = 1.2   -> about 1 partition
```

A clustering key only helps predicates on its leading columns. If a workload filters on `customer_id` alone, a key of `(order_date, customer_id)` may not prune much, because the date ranges of neighboring partitions still overlap.

## How do you read a query profile and tune a slow query?

The query profile is the execution record of one statement, and it is the single most useful tuning artifact in Snowflake. Open Query History in Snowsight, pick the expensive query, and you get a tree of operators — scans, joins, aggregations, sorts — annotated with how many partitions were scanned out of the total, how many bytes and rows moved, how long each operator took, and how much data spilled to disk.

Read it top-down for shape and bottom-up for cost. The overall statistics show total duration split into compilation, queuing, and execution. The operator tree shows which step dominated. A `TableScan` with `partitions scanned` close to `partitions total` is a pruning problem. A `Join` producing far more rows than either input is an exploding join. A `Sort` or `Aggregate` with bytes spilled is a memory problem. The most-offending operator is almost always where you start.

`QUERY_HISTORY` gives you the same numbers in SQL so you can find candidates without clicking. Useful columns are `TOTAL_ELAPSED_TIME`, `QUEUED_OVERLOAD_TIME`, `QUEUED_PROVISIONING_TIME`, `COMPILATION_TIME`, `EXECUTION_TIME`, `PARTITIONS_SCANNED`, `PARTITIONS_TOTAL`, `PERCENTAGE_SCANNED_FROM_CACHE`, the two spill columns, and `CREDITS_USED_CLOUD_SERVICES`. Filter to a time window and sort by elapsed time, then drill into the individual query.

```sql
-- 1. Find the worst offenders in the last day.
SELECT query_id, query_type, warehouse_name, warehouse_size, user_name,
       total_elapsed_time / 1000 AS elapsed_s,
       queued_overload_time / 1000 AS queued_s,
       compilation_time / 1000 AS compile_s,
       execution_time / 1000 AS exec_s,
       partitions_scanned, partitions_total,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       percentage_scanned_from_cache,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage,
       credits_used_cloud_services
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
ORDER BY total_elapsed_time DESC
LIMIT 25;

-- 2. Inspect one query in detail, including its text.
SELECT query_id, query_text, total_elapsed_time, execution_time,
       partitions_scanned, partitions_total,
       bytes_scanned, bytes_spilled_to_local_storage, bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE query_id = '<query_id>';

-- 3. Look at the plan, not just the timings.
SELECT SYSTEM$EXPLAIN_PLAN_JSON('<query_id>');
```

A query profile tells you which operator dominated and whether the cause was pruning, spilling, queuing, or query shape — read the numbers from `QUERY_HISTORY` first, then confirm in the operator tree before changing anything.

The profile is a tree, and a slow child makes every ancestor look slow, because a parent's elapsed time includes the child's. Read it bottom-up: find the operator with the largest share of execution time, fix that node's input, then re-read. The operator with the most rows is not necessarily the one to tune.

Compare the numbers in order rather than all at once: `QUEUED_OVERLOAD_TIME` versus `EXECUTION_TIME` (concurrency or SQL), `PARTITIONS_SCANNED` divided by `PARTITIONS_TOTAL` (pruning), `PERCENTAGE_SCANNED_FROM_CACHE` (warm or cold), and the two spill columns (memory). Only when queuing is negligible and the scan is already small is the remaining time a pure operator-cost problem.

`TOTAL_ELAPSED_TIME` includes compilation and queuing, so a query can look slow on a dashboard while the SQL itself is fast. Subtract the pieces before rewriting anything.

## How do you use EXPLAIN and the query profile for pruning?

`EXPLAIN` compiles a statement and returns the plan without running it, so it is cheap and safe to use while iterating. `EXPLAIN USING TEXT` gives a compact operator listing that is easy to skim in a terminal; `EXPLAIN USING JSON` gives the full plan object with filters, join order, and estimated partition counts. For a query that already ran, `SYSTEM$EXPLAIN_PLAN_JSON('<query_id>')` returns the plan that actually executed.

The two things you are checking are filter placement and scan size. Filters should appear on the scan node, not on a node far above it after a join has already materialized rows, because a filter pushed to the scan lets the optimizer prune partitions and reduce rows early. The scan node should estimate a small number of partitions relative to the table total for a selective predicate. If it estimates every partition, no amount of warehouse size will help.

Compare the estimate to reality. If `EXPLAIN` predicts a handful of partitions but the profile shows a full scan, the predicate was not usable for pruning, often because of a cast, a function, a type mismatch, or a `LIKE` with a leading wildcard. If both agree on a full scan, the table is genuinely unordered for that predicate and a clustering key or a sorted reload is the fix.

```sql
-- Compact plan: look for filters on the scan node.
EXPLAIN USING TEXT
SELECT customer_id, SUM(amount)
FROM analytics.orders
WHERE order_date BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY customer_id;

-- Full plan object with estimated partitions per scan.
EXPLAIN USING JSON
SELECT COUNT(*) FROM analytics.events WHERE event_date = '2026-10-04';

-- The plan that actually ran for a finished query.
SELECT SYSTEM$EXPLAIN_PLAN_JSON('<query_id>');

-- Confirm what the run actually scanned.
SELECT partitions_scanned, partitions_total, bytes_scanned,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned
FROM snowflake.account_usage.query_history
WHERE query_id = '<query_id>';
```

`EXPLAIN` shows the plan before you pay for it: confirm filters sit on the scan, the estimated partition count is small, and the executed profile agrees before you reach for a bigger warehouse.

`EXPLAIN` compiles without executing, so the plan costs cloud-services work rather than warehouse work and does not start a warehouse. It also means `EXPLAIN` cannot show runtime facts such as actual rows, spill, or cache behavior; those exist only in the profile of a query that actually ran.

A useful pattern is to run `EXPLAIN USING JSON` for the estimate, then run the query once and compare the scan node's estimated partition count with `PARTITIONS_SCANNED` in `QUERY_HISTORY`. Agreement means the optimizer understood the predicate; disagreement usually means a cast, a function, or a `LIKE` pattern that prevented pruning.

`EXPLAIN` reflects current metadata, so it can be optimistic on a table whose data changed since statistics were last refreshed, and it does not prove the warehouse has enough memory to avoid spilling.

## How do you size a virtual warehouse?

Warehouse size trades latency, concurrency, and credits. Doubling the size doubles the number of compute nodes and roughly doubles the credit rate per hour, so a query that parallelizes well finishes in about half the time for about the same total credits. Sizing is therefore mostly about how long you are willing to wait, and about how many queries must run at once, not about whether the work gets cheaper.

Size up when a single query is slow because it has a lot of parallelizable work: large scans, big joins, wide aggregations, sorts that spill. Size out — add clusters with a multi-cluster warehouse — when many concurrent users queue for the same compute. Do not size up to fix a query that scans every partition, waits on a lock, or looks up a single row; in those cases the extra nodes sit idle and you pay for them anyway.

Choose by workload shape rather than by habit. Dashboards and development usually fit XSMALL or SMALL. Standard ELT and mixed concurrency fit MEDIUM or LARGE. Large joins and backfills justify XLARGE. Sizes beyond that are rare and usually indicate a query problem. Because billing is per second with a 60-second minimum, a warm MEDIUM warehouse serving steady traffic is often cheaper than many XSMALL warehouses starting and stopping.

The gotcha is that small warehouses are slower per query but cost the same per credit-hour, so per unit of work a well-tuned small warehouse can be cheaper. If a SMALL warehouse runs a query in 10 seconds and a MEDIUM runs it in 5, both consume roughly the same credits. The difference is latency and concurrency headroom. Measure with `WAREHOUSE_METERING_HISTORY` and `QUERY_HISTORY` before committing to a size.

```sql
-- Start small, work up only when a measured query is latency-bound.
CREATE OR REPLACE WAREHOUSE bi_wh WAREHOUSE_SIZE = 'SMALL'
  AUTO_SUSPEND = 300 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;

ALTER WAREHOUSE bi_wh SET WAREHOUSE_SIZE = 'MEDIUM';   -- if p95 latency is unacceptable
ALTER WAREHOUSE bi_wh SET WAREHOUSE_SIZE = 'SMALL';    -- and back down after measuring

-- Credits and runtime per warehouse over the last week.
SELECT warehouse_name,
       SUM(credits_used) AS credits,
       SUM(credits_used_compute) AS compute_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;

-- Average and p95 query latency per warehouse.
SELECT warehouse_name,
       ROUND(AVG(execution_time) / 1000, 1) AS avg_exec_s,
       ROUND(APPROX_PERCENTILE(execution_time, 0.95) / 1000, 1) AS p95_exec_s,
       COUNT(*) AS queries
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY p95_exec_s DESC;
```

```text
XS    =  1 credit/hour    dashboards, small transforms, dev
S     =  2 credits/hour   light BI, moderate dbt models
M     =  4 credits/hour   standard ELT, mixed concurrency
L     =  8 credits/hour   large joins, big backfills
XL    = 16 credits/hour   heavy transformations
2XL+  = 32+ credits/hour  rare; usually a signal to fix the query
```

Size up for latency and scale out for concurrency, but expect roughly constant credits per unit of parallel work — so fix pruning and query shape before paying for a bigger warehouse.

Scale-up only accelerates the parts of a plan that can be spread across nodes. A skewed join key, a single-threaded aggregation, or one very wide row concentrates work and leaves extra nodes idle. Before resizing, check whether the dominant operator is parallel or serial; if it is serial, a bigger warehouse adds cost without adding speed.

A query that runs 600 seconds on MEDIUM (4 credits/hour) consumes 4 x 600 / 3600 = 0.667 credits. Run in 300 seconds on LARGE (8 credits/hour), the same query consumes 8 x 300 / 3600 = 0.667 credits. Scaling up is credit-neutral when the runtime halves; it is a loss when the runtime falls by less than the rate rises.

A resize applies to statements that start after it completes; a query already running keeps the size and resource pool it started with. Resize between batches, not in the middle of one.

## What are multi-cluster warehouses and when do you use them?

A multi-cluster warehouse is a single logical warehouse backed by several identical clusters. Snowflake starts additional clusters when queries are waiting and shuts them down when demand falls. From the user's point of view there is one warehouse name and one connection string; the platform handles the fan-out. This is the scale-out answer to concurrency, in contrast to resizing, which is scale-up.

Two settings matter. `MIN_CLUSTER_COUNT` and `MAX_CLUSTER_COUNT` bound the fleet; setting `MIN_CLUSTER_COUNT` above 1 keeps clusters warm at a standing cost, which some interactive workloads accept. `SCALING_POLICY` controls how eagerly clusters are added and removed. `STANDARD` adds clusters quickly when queries queue and removes them when they are idle, which suits interactive BI with unpredictable bursts. `ECONOMY` adds clusters more slowly and keeps them roughly six minutes after going idle, which suits batch workloads that would rather wait than pay for churn.

Use multi-cluster when several concurrent users or jobs run similar queries on the same warehouse and `QUEUED_OVERLOAD_TIME` is a meaningful share of `TOTAL_ELAPSED_TIME`. If queries are slow because each one scans a lot of data, the fix is a bigger cluster or a better query, not more clusters. If queueing is the problem, adding clusters raises throughput without changing per-query latency. Multi-cluster warehouses require Enterprise Edition.

```sql
ALTER WAREHOUSE bi_wh SET
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 6
  SCALING_POLICY = 'STANDARD';

ALTER WAREHOUSE reporting_wh SET
  MIN_CLUSTER_COUNT = 2
  MAX_CLUSTER_COUNT = 10
  SCALING_POLICY = 'ECONOMY';

-- Queueing is the signal that scale-out is worth trying.
SELECT warehouse_name,
       COUNT(*) AS queries,
       SUM(CASE WHEN queued_overload_time > 0 THEN 1 ELSE 0 END) AS queued_queries,
       ROUND(SUM(queued_overload_time) / 1000, 1) AS total_queued_s
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY total_queued_s DESC;

-- Watch cluster count and queueing over time for one warehouse.
SELECT start_time, warehouse_name, avg_running, avg_queued_load
FROM TABLE(information_schema.warehouse_load_history(
  warehouse_name => 'BI_WH',
  date_range_start => DATEADD('hour', -12, CURRENT_TIMESTAMP())));
```

Multi-cluster warehouses absorb concurrency by adding clusters, so reach for them when queue time — not per-query scan cost — is the bottleneck, and pick `STANDARD` for interactive work and `ECONOMY` for batch.

Each cluster is a full copy of the warehouse and bills at the same per-hour rate, so `MAX_CLUSTER_COUNT` is a cost multiplier as well as a capacity setting. Clusters are added when queries are waiting and removed when they are not, subject to `MIN_CLUSTER_COUNT`. `MAX_CONCURRENCY_LEVEL` is the number of concurrent queries a single cluster is expected to absorb; a lower value makes Snowflake add the next cluster sooner.

STANDARD is reactive: it adds a cluster as soon as queries queue and removes clusters when load drops, which keeps interactive latency low. ECONOMY is deliberate: it waits for evidence that an extra cluster will be used for a while before adding it, so a burst may queue slightly longer but the warehouse avoids starting and stopping clusters repeatedly.

Adding clusters does not make a single query faster; every query still runs inside one cluster. If one query is slow with no queueing, multi-cluster is the wrong lever.

## How do auto-suspend and auto-resume affect cost?

`AUTO_SUSPEND` shuts a warehouse down after a period of inactivity and `AUTO_RESUME` starts it again when the next query arrives. Together they are the simplest and most effective cost control in Snowflake, because compute is billed only while the warehouse is running. Leaving `AUTO_SUSPEND` at 0 means the warehouse never suspends, so you pay for every idle hour.

The correct suspend value is a tradeoff between credit savings and cache warmth. A warehouse that suspends loses its local SSD cache, so the next query reads table data from storage instead of cache. For batch pipelines that run on a schedule and then sit idle, 60 seconds is usually right: the next run will not benefit from a cache that is an hour old anyway. For interactive BI with users running queries throughout the day, 300 to 600 seconds avoids constant cold starts and cache rebuilds.

`AUTO_RESUME` should almost always be `TRUE`. Disabling it saves nothing because a suspended warehouse costs nothing, and it introduces a failure mode where a scheduled task or a dashboard query arrives at a suspended warehouse and errors instead of waking it. The exceptions are deliberate: an administrative warehouse you never want started by accident.

The gotcha is that a very short auto-suspend on a busy warehouse can cost more than it saves, because each restart pays the 60-second minimum and loses cache. Conversely, a long auto-suspend on a warehouse used only twice a day wastes credits between runs. Inspect actual start and stop events before choosing, and prefer several purpose-built warehouses over one shared always-on warehouse.

```sql
ALTER WAREHOUSE bi_wh        SET AUTO_SUSPEND = 600 AUTO_RESUME = TRUE;  -- interactive bursts
ALTER WAREHOUSE elt_wh       SET AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;  -- scheduled batch
ALTER WAREHOUSE reporting_wh SET AUTO_SUSPEND = 0;                        -- costs idle credits

-- When did each warehouse start, suspend, or resize?
SELECT warehouse_name, previous_state, state, timestamp, user_name
FROM snowflake.account_usage.warehouse_events_history
WHERE timestamp >= DATEADD('day', -2, CURRENT_TIMESTAMP())
ORDER BY timestamp DESC
LIMIT 50;

-- Idle time is visible as long running stretches with no queries.
SELECT warehouse_name,
       SUM(credits_used) AS credits,
       COUNT(DISTINCT DATE_TRUNC('hour', start_time)) AS active_hours
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -3, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;
```

Auto-suspend is free money as long as you match the value to the workload: 60 seconds for batch, several minutes for interactive, never 0 — and keep auto-resume on so scheduled work cannot fail silently.

Auto-suspend is measured in seconds of warehouse inactivity, and the 60-second billing minimum applies to each start, so a warehouse that wakes for a five-second statement is billed for a minute. A multi-cluster warehouse pays that minimum per cluster, and resuming a warehouse also means its local SSD cache starts over.

A SMALL warehouse (2 credits/hour) left running through a 14-hour overnight gap consumes 2 x 14 = 28 idle credits. With `AUTO_SUSPEND = 60` it consumes roughly nothing after the last statement, at the cost of a cold cache on the next run.

Values below 60 seconds are effectively the 60-second minimum for a standard warehouse, so there is no benefit in setting an aggressive number. What matters is matching the value to how likely the next query is.

## What caching layers does Snowflake have?

Snowflake caches at three levels, and naming them precisely is the point of the question. The result cache holds the complete output of a query, keyed by the exact query text and the current versions of every table it touched. It is valid for 24 hours and is served by cloud services without starting a warehouse, so a repeated dashboard refresh can be effectively free.

The local disk cache, often called the warehouse cache or SSD cache, holds raw table data that a warehouse has recently read, on the local storage of its nodes. Queries that re-read the same micro-partitions hit this cache and avoid fetching from object storage. It belongs to one warehouse and disappears when that warehouse is suspended.

The remote cache, also called the metadata cache or the persistent cache for table data, lives in cloud services and holds micro-partition metadata and some file headers. It persists across warehouse suspensions and is shared by all warehouses in the account, so a first scan after a cold start is cheaper than it would otherwise be. Most cloud-services work is included at no charge.

The gotchas are worth memorizing. The result cache is invalidated by any DML on the underlying tables, and it is bypassed by non-deterministic functions such as `CURRENT_TIMESTAMP()` or `RANDOM()`. It also requires identical query text, so a stray comment or whitespace difference is a different query. If `PERCENTAGE_SCANNED_FROM_CACHE` is low for a query you expected to be cached, check whether the table changed or the query text varies.

```sql
-- How much of each query was served from cache, and were partitions even scanned?
SELECT query_id, warehouse_name, percentage_scanned_from_cache,
       partitions_scanned, partitions_total, bytes_scanned
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -3, CURRENT_TIMESTAMP())
ORDER BY start_time DESC
LIMIT 20;

-- Non-deterministic: never result-cached.
SELECT CURRENT_TIMESTAMP(), COUNT(*) FROM analytics.orders;

-- Deterministic and identical each time: eligible for the result cache.
SELECT COUNT(*) FROM analytics.orders;

-- Compare cache-aware timings across two runs of the same text.
SELECT query_id, total_elapsed_time / 1000 AS elapsed_s, percentage_scanned_from_cache
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE 'SELECT COUNT(*) FROM analytics.orders%'
ORDER BY start_time DESC
LIMIT 5;
```

Know the three caches — result cache (24 hours, free, invalidated by DML), warehouse SSD cache (lost on suspend), and the shared remote metadata cache (persistent) — and check `PERCENTAGE_SCANNED_FROM_CACHE` before assuming a query is cacheable.

The result cache is keyed by query text, the role and its privileges, the relevant session parameters, and the versions of the tables involved. It lives in cloud services and needs no warehouse, which is why a hit can return while the warehouse is suspended. The warehouse cache is raw micro-partition bytes on the nodes' local SSD and belongs to one warehouse. The metadata cache is shared account-wide and survives suspension.

```sql
SELECT COUNT(*) FROM analytics.orders;      -- runs, result stored
SELECT COUNT(*) FROM analytics.orders;      -- identical text, unchanged tables: cache hit
INSERT INTO analytics.orders (order_id) VALUES (999999);  -- new table version
SELECT COUNT(*) FROM analytics.orders;      -- re-executed even though the text is identical
```

A cache hit is invisible in `PARTITIONS_SCANNED` because nothing is scanned, so a `PERCENTAGE_SCANNED_FROM_CACHE` of zero can still mean the result cache answered the query. Check `TOTAL_ELAPSED_TIME` and whether a warehouse started before concluding the cache missed.

## What is Time Travel and Fail-safe?

Time Travel lets you query, clone, or restore data as it existed earlier by referencing historical table versions. You can pin a timestamp, an offset in seconds, or a specific statement id, and because Snowflake never overwrites micro-partitions in place, the old versions are simply still addressable. The retention window is configurable per table, schema, or database with `DATA_RETENTION_TIME_IN_DAYS`: 0 to 1 day on Standard Edition and up to 90 days on Enterprise Edition.

Fail-safe is the seven-day period after Time Travel expires. During Fail-safe data is not queryable by users; only Snowflake support can recover it, and doing so is a support operation rather than a self-service query. Fail-safe is not configurable and always seven days, and it applies to permanent tables but not to transient or temporary tables.

Time Travel is genuinely useful for recovering from a bad `DELETE`, reproducing a report as of a reporting date, and taking a consistent snapshot with a zero-copy clone. It also underpins streams, because a stream needs the source table's historical versions to compute a delta position.

The gotcha is treating Time Travel as backup. It is retention-bound, so anything older than the window is gone unless you cloned it; extended retention increases storage cost because old partitions are still billed; and Fail-safe is a last resort, not a restore plan. Real disaster recovery needs database replication to another region and, ideally, periodic exports or clones to a separate account.

```sql
ALTER TABLE analytics.orders SET DATA_RETENTION_TIME_IN_DAYS = 30;

-- Query data as of an absolute time, an offset, or after a specific statement.
SELECT COUNT(*) FROM analytics.orders AT (TIMESTAMP => '2026-10-01 00:00:00'::TIMESTAMP);
SELECT COUNT(*) FROM analytics.orders AT (OFFSET => -3600);
SELECT COUNT(*) FROM analytics.orders BEFORE (STATEMENT => '<query_id>');

-- Recover a dropped object or snapshot a table without copying it.
UNDROP TABLE analytics.orders;
CREATE TABLE analytics.orders_audit CLONE analytics.orders AT (OFFSET => -86400);

-- How much history a table keeps, and how much storage old versions consume.
SELECT table_name, retention_time, dropped, created
FROM analytics.information_schema.tables
WHERE table_name = 'ORDERS';
```

Time Travel is a queryable safety net bounded by retention, Fail-safe is a seven-day support-only lifeline, and neither replaces replication and exports for real disaster recovery.

`AT` and `BEFORE` accept a timestamp, an offset in seconds, or a statement id. `BEFORE (STATEMENT => id)` returns the state just before that statement ran and excludes its effects; `AT (STATEMENT => id)` includes them. `UNDROP` restores a dropped object while its retention window is open, and a clone taken `AT` a timestamp is a durable way to keep a point-in-time copy after retention passes.

Retention is configurable at database, schema, and table level. Standard Edition allows 0 or 1 day; Enterprise Edition allows 0 to 90 days. Fail-safe adds a fixed seven days after Time Travel ends, applies to permanent tables only, and is not queryable: recovery requires Snowflake support. Transient and temporary tables get at most one day of Time Travel and no Fail-safe at all.

Longer retention is not free. Superseded micro-partitions remain stored for the whole window, so a heavily changing table with 90-day retention carries a storage bill proportional to that history. Transient tables are the cheaper choice for rebuildable staging where Fail-safe adds no value.

## What is zero-copy cloning?

A clone creates a new table, schema, or database that initially shares all of its micro-partitions with the source. Only metadata is copied, so a clone of a huge database finishes in seconds and stores almost nothing extra at the moment it is created. As the clone and the source diverge through DML, new partitions are written and only those new partitions are billed.

Clones can be taken at a point in time, which combines naturally with Time Travel: `CREATE TABLE ... CLONE ... AT (TIMESTAMP => ...)` gives you an immutable snapshot for audit or regression testing. Because the source and clone are independent objects after creation, changes to one never affect the other, and dropping the source does not invalidate a clone or vice versa.

The everyday use is development and testing. Clone production into a development database, run a migration or a dbt full refresh against the clone, compare results, and swap or drop it. The same pattern supports release testing, performance experiments on representative data, and ad-hoc analysis without touching production tables.

The gotcha is privileges. A clone inherits access through the object hierarchy (database, schema, and the clone's own grants), not the source object's grants, so a team that could read the source may not automatically read the clone. Also remember that storage savings shrink as the clone diverges: if a clone is heavily modified it eventually costs about as much as a copy.

```sql
-- Clone an entire database for a safe development sandbox.
CREATE DATABASE analytics_dev CLONE analytics;

-- Snapshot one table at a point in time.
CREATE TABLE analytics.orders_snapshot CLONE analytics.orders
  AT (TIMESTAMP => '2026-10-01 00:00:00'::TIMESTAMP);

-- The clone is independent as soon as its partitions are modified.
UPDATE analytics_dev.marts.daily_revenue
SET revenue = 0
WHERE order_date < '2026-01-01';

-- Grants must be applied to the clone explicitly.
GRANT USAGE ON DATABASE analytics_dev TO ROLE analytics_engineer;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics_dev.marts TO ROLE analytics_engineer;

DROP DATABASE analytics_dev;
```

Zero-copy clones share micro-partitions until modified, which makes production-sized dev, test, and audit environments fast and cheap — just re-grant access, because privileges do not travel with the clone.

A clone copies the object's metadata and records a reference to each of the source's micro-partitions. Both objects read the same files until one of them changes them. Because partitions are immutable, a change on either side writes a new partition for the affected rows rather than editing a shared one, so the two objects diverge only where they differ. This is copy-on-write at partition granularity, not at row granularity.

A 100 TB database clones in seconds and adds almost no storage. If the clone rewrites 5 percent of its partitions during a test, it adds roughly 5 percent of the data, not 100 percent. A clone that is modified everywhere converges toward the cost of a full copy.

Cloning shares storage, so dropping the source while the clone exists frees nothing. A clone also starts its own Time Travel history when it is created; it does not inherit the source's history, so clone again if you need a later retention window.

## What are streams and how do they capture CDC?

A stream is a change-tracking object over a table, view, or external table. Once created, it records inserts, updates, and deletes relative to the last time the stream was consumed, exposing them as rows with `METADATA$ACTION`, `METADATA$ISUPDATE`, and `METADATA$ROW_ID` columns. It is Snowflake's native change data capture primitive and the usual way to build incremental pipelines without re-scanning whole tables.

The mechanism is transactional and offset-based. Reading a stream with a plain `SELECT` does not advance it; the offset moves only when the stream is consumed inside a DML statement such as `INSERT`, `MERGE`, or `CREATE TABLE AS`. That means a stream and its consumer commit together, so a failed load does not silently skip changes, and the same delta is not applied twice by a single successful statement.

A typical pattern merges the delta into a target using `QUALIFY ROW_NUMBER()` to collapse multiple changes to the same key within one batch, which is important because a key can be updated more than once between reads. Streams are also the trigger for tasks: `SYSTEM$STREAM_HAS_DATA` gates a task so it only runs when there is something to process.

The gotcha is staleness. A stream tracks changes only while the source table's historical versions are retained, so if a stream is not consumed within the table's data retention window it becomes stale and errors on the next read. Snowflake temporarily extends retention for unconsumed streams up to `MAX_DATA_EXTENSION_TIME_IN_DAYS` (14 days by default). Also note that a stream gives you a delta, not idempotency: write the consumer so re-running a failed batch is safe.

```sql
CREATE OR REPLACE STREAM raw.orders.orders_stream ON TABLE raw.orders.orders;

-- Inspect the pending delta without consuming it.
SELECT METADATA$ACTION, METADATA$ISUPDATE, METADATA$ROW_ID,
       order_id, status, updated_at
FROM raw.orders.orders_stream
LIMIT 10;

-- Consuming the stream in DML advances the offset atomically with the write.
MERGE INTO analytics.orders AS t
USING (
  SELECT order_id, status, updated_at, METADATA$ACTION AS action
  FROM raw.orders.orders_stream
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1
) AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED AND s.action = 'INSERT'
  THEN INSERT (order_id, status, updated_at) VALUES (s.order_id, s.status, s.updated_at);

-- Check whether a stream has pending data before starting work.
SELECT SYSTEM$STREAM_HAS_DATA('raw.orders.orders_stream');
SHOW STREAMS LIKE 'ORDERS_STREAM' IN SCHEMA raw.orders;
```

A stream is a transactional change cursor over a table: it only advances when consumed by DML, it expires if you leave it unread past the retention window, and it hands you a delta that you must merge idempotently.

A stream does not copy rows; it holds an offset into the source table's change history and a state flag. When a DML statement reads the stream and writes the result, the offset advances inside the same transaction, so a rollback leaves the stream pointing at the same changes. A `SELECT` from the stream alone never advances it, no matter how many times it runs.

Standard streams capture inserts, updates, and deletes, exposing `METADATA$ACTION`, `METADATA$ISUPDATE`, and `METADATA$ROW_ID`. Append-only streams capture inserts only and do less bookkeeping, so they are cheaper for immutable event tables where rows are never updated or deleted. Choose append-only unless you genuinely need update and delete tracking.

A stream goes stale when the source's data retention window passes a point it still needs. `MAX_DATA_EXTENSION_TIME_IN_DAYS` (14 by default) extends retention for unconsumed streams, and once a stream is stale it cannot be recovered: it must be dropped and recreated, losing every change in between.

## What are tasks and how do you build incremental pipelines?

A task runs a SQL statement, a stored procedure call, or a Snowflake Scripting block either on a schedule or after another task. Tasks are the orchestration primitive inside Snowflake: you chain them with `AFTER` to form a DAG, and each task can run on its own warehouse or serverlessly if you omit `WAREHOUSE`. This is how a pipeline stays inside the platform instead of depending on an external scheduler.

The canonical incremental pattern combines a stream with a task. The task is gated by `WHEN SYSTEM$STREAM_HAS_DATA('<stream>')`, so it only runs when there is a delta, and its body consumes the stream in a `MERGE` or `INSERT`. You create the whole DAG, then resume the children before the root (`ALTER TASK ... RESUME` walks upward), because a task cannot be resumed while its parent is suspended.

Serverless tasks are worth knowing about: omit `WAREHOUSE` and Snowflake provisions compute per run, which avoids paying for an idle warehouse between infrequent executions. Scheduled tasks support cron-like `SCHEDULE` expressions as well as intervals, and `SYSTEM$TASK_DEPENDENTS_ENABLE` can resume a whole DAG at once. Task history is queryable, so failures and durations are observable.

The gotchas are retries and overlapping runs. Tasks can overlap if a run takes longer than its schedule, and a task tree does not roll back as a unit: if a child fails, already-committed parents stay committed. Design each step to be idempotent and safe to retry, keep tasks small, and use `TASK_HISTORY` to watch for failures and long-running steps.

```sql
CREATE OR REPLACE TASK raw.orders.load_task
  WAREHOUSE = elt_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw.orders.orders_stream')
AS CALL analytics.load_orders_incremental();

CREATE OR REPLACE TASK analytics.transform_task
  WAREHOUSE = transform_wh
  AFTER raw.orders.load_task
AS CALL analytics.build_orders_marts();

-- Child tasks first, then the root.
ALTER TASK analytics.transform_task RESUME;
ALTER TASK raw.orders.load_task RESUME;

-- Serverless alternative: no warehouse to keep warm.
CREATE OR REPLACE TASK analytics.compact_task
  SCHEDULE = 'USING CRON 0 3 * * * America/Chicago'
AS INSERT INTO analytics.audit_log SELECT CURRENT_TIMESTAMP(), 'compacted';

-- Observe runs, durations, and errors.
SELECT name, state, scheduled_time, completed_time, error_message
FROM TABLE(information_schema.task_history(
  scheduled_time_range_start => DATEADD('day', -1, CURRENT_TIMESTAMP())))
ORDER BY scheduled_time DESC
LIMIT 50;
```

Tasks chain SQL into a DAG and, gated on `SYSTEM$STREAM_HAS_DATA`, give you an incremental pipeline inside Snowflake — make every step idempotent, because a failed subtree does not roll back its parents.

A standalone task has a schedule; a child task has `AFTER` and no schedule. A run of the root cascades to children as predecessors succeed. Task graphs are versioned, so a run completes against the definition that existed when it started and editing a task mid-run does not change an executing graph. `EXECUTE TASK <name>` triggers a manual run of a standalone task or the root of a graph, independent of the schedule; it runs the root even when suspended but does not resume suspended child tasks.

Serverless tasks omit `WAREHOUSE` and run on Snowflake-managed compute, billed as serverless credits visible in `SERVERLESS_TASK_HISTORY`; warehouse-backed tasks consume the named warehouse instead and can be gated with `WHEN SYSTEM$STREAM_HAS_DATA(...)` so an idle schedule costs nothing. Scheduled tasks accept a plain interval or a cron expression with a time zone.

Because a graph is not one transaction, a failed child leaves its committed predecessors applied and a retry re-runs only from the failed point. Every task body should therefore be idempotent, and overlap should be handled deliberately so a slow run and the next scheduled run do not fight over the same target rows.

## What are dynamic tables?

A dynamic table is a declarative, continuously refreshed table defined by a query. Instead of writing the incremental logic yourself, you state the result you want and a freshness target with `TARGET_LAG`, and Snowflake decides when to refresh and how to incrementalize the query. It is the closest thing Snowflake has to a materialized view that can include joins and arbitrary transformations.

The freshness model is the important idea. `TARGET_LAG` says how far behind the base tables the dynamic table is allowed to be, for example `'10 minutes'` or `DOWNSTREAM`. When one dynamic table feeds another, `DOWNSTREAM` on the upstream table lets Snowflake schedule the chain so the final table meets its own lag without refreshing intermediates more often than necessary. Refreshes run on the warehouse you name and use incremental computation where the query allows it.

Dynamic tables are a good fit when a pipeline can be expressed entirely as SQL, when near-real-time freshness matters more than exact scheduling, and when you want Snowflake to manage dependencies between stages. They are a poor fit for procedural logic, for precisely timed loads, and for transformations the optimizer cannot incrementalize, which degrade to full refreshes.

The gotcha is cost opacity. You are trading control for convenience: a short target lag on a large query can refresh often and consume significant credits, and a query that cannot be incrementalized recomputes from scratch every cycle. Monitor refresh history with `DYNAMIC_TABLE_REFRESH_HISTORY` and watch that the lag you asked for is the lag you actually get.

```sql
CREATE OR REPLACE DYNAMIC TABLE analytics.daily_revenue
  TARGET_LAG = '10 minutes'
  WAREHOUSE = transform_wh
AS
  SELECT o.order_date, c.segment,
         SUM(o.amount) AS revenue,
         COUNT(*) AS orders
  FROM analytics.orders o
  JOIN analytics.customers c USING (customer_id)
  GROUP BY o.order_date, c.segment;

-- Chained dynamic table: DOWNSTREAM lets Snowflake schedule the whole chain.
CREATE OR REPLACE DYNAMIC TABLE analytics.segment_revenue
  TARGET_LAG = '10 minutes'
  WAREHOUSE = transform_wh
AS
  SELECT segment, SUM(revenue) AS revenue
  FROM analytics.daily_revenue
  GROUP BY segment;

SELECT name, target_lag_sec, scheduling_state, latest_data_timestamp
FROM TABLE(information_schema.dynamic_tables());

-- Actual refresh duration and state per run.
SELECT name, state, refresh_start_time, refresh_end_time, data_timestamp
FROM TABLE(information_schema.dynamic_table_refresh_history(
  name => 'analytics.daily_revenue',
  data_timestamp_start => DATEADD('day', -1, CURRENT_TIMESTAMP())));
```

Dynamic tables let you declare the result and a `TARGET_LAG` and let Snowflake schedule the refresh, which is powerful for SQL-shaped pipelines but hides cost — always watch refresh history and the lag you actually achieve.

A dynamic table stores its own data and keeps a refresh history. The default `REFRESH_MODE = AUTO` lets Snowflake choose incremental refresh when the query supports it and a full refresh when it does not; `INCREMENTAL` or `FULL` can be forced, and a full refresh recomputes the whole result. `TARGET_LAG = DOWNSTREAM` means the table has no independent freshness promise and is scheduled to keep its consumers on time.

Dynamic tables form a dependency graph, so adding a new downstream table does not require editing upstream schedules. That managed graph is the main operational advantage over streams plus tasks, where every new consumer means another task to create, wire, and resume in the right order.

`TARGET_LAG` is a target, not a guarantee. A refresh that takes longer than the lag, or a query that cannot be incrementalized and must fully recompute, leaves the table reliably behind. Watch `DYNAMIC_TABLE_REFRESH_HISTORY` for refresh duration and state, and compare the achieved data timestamp against the requested lag.

## What is Snowpipe and when do you use it?

Snowpipe is Snowflake-managed continuous ingestion from files in a stage. A pipe wraps a `COPY INTO`, and new files trigger the pipe through cloud event notifications (S3, GCS, Azure) or the REST API, rather than waiting for a scheduled load. You do not need a running warehouse for the load: Snowpipe uses Snowflake-managed compute and is billed per file handled, which is why it suits steady, low-latency file arrivals.

The alternative at the other end of the spectrum is scheduled batch loading with `COPY INTO` on a warehouse. Batch is cheaper per byte when files are large and arrive in predictable waves, because the warehouse can load many files in parallel and then suspend. Snowpipe is better when files arrive continuously and you want them queryable within a minute or two without paying for an always-on loader.

For row-level latency, Snowpipe Streaming bypasses files entirely. A client writes rows through the Snowpipe Streaming API or the Kafka connector directly into a table, which removes the file-landing latency and the small-file problem. It is the right tool for application events that must land in seconds.

The gotchas are delivery semantics and duplicate handling. Snowpipe and Snowpipe Streaming are both at-least-once, so a file or a row can be delivered more than once; dedupe on a business key or a file name. Ingesting thousands of very small files is the classic ingestion anti-pattern and hurts both Snowpipe and batch loads, so compact files before loading where you can, and monitor `COPY_HISTORY`/`PIPE_USAGE_HISTORY` for errors and cost.

```sql
-- File-based continuous ingestion with cloud event notifications.
CREATE OR REPLACE PIPE raw.orders.orders_pipe AUTO_INGEST = TRUE
AS COPY INTO raw.orders.orders
   FROM @raw.orders.orders_landing
   FILE_FORMAT = (TYPE = PARQUET)
   MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

-- Pipe state and the notification channel used by AUTO_INGEST.
SELECT SYSTEM$PIPE_STATUS('raw.orders.orders_pipe');
SHOW PIPES LIKE 'ORDERS_PIPE' IN SCHEMA raw.orders;

-- What Snowpipe loaded recently, and what failed.
SELECT file_name, row_count, error_count, status, last_load_time
FROM TABLE(information_schema.copy_history(
  table_name => 'raw.orders.orders',
  start_time => DATEADD('hour', -24, CURRENT_TIMESTAMP())));

-- Serverless credits consumed by pipes.
SELECT pipe_name, SUM(credits_used) AS credits
FROM snowflake.account_usage.pipe_usage_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY pipe_name
ORDER BY credits DESC;
```

Use Snowpipe for continuous file arrival, scheduled `COPY INTO` for large predictable batches, and Snowpipe Streaming for row-level latency — but plan for at-least-once delivery and avoid thousands of tiny files.

A pipe is a stored `COPY INTO` plus a load-history state. With `AUTO_INGEST = TRUE`, Snowflake reports a notification channel (an SQS queue, a Pub/Sub subscription, or an Event Grid endpoint) that you register with the bucket; an object-created event then calls the pipe, which loads just that file. Snowflake records each file in load metadata so a redelivered event does not load it again.

Snowpipe runs on Snowflake-managed compute billed as serverless credits, and every file carries its own event, metadata, and processing overhead, so thousands of tiny files multiply both latency and cost. Snowpipe Streaming avoids files entirely: a client writes rows through the API and Snowflake commits them into the table, trading per-file overhead for row-level latency.

Delivery is at-least-once. A file or a row can be processed twice, so the target table or the downstream merge must key on a business key or the file name. The pipe will not make duplicates safe for you.

## What are stages and how do you load with COPY INTO?

A stage is a named location Snowflake loads from or unloads to. An internal stage is managed by Snowflake inside your account; an external stage points at your own S3 bucket, GCS bucket, or Azure container through a storage integration, so no credentials live in the stage definition. User stages (`@~`) and table stages (`@%table`) exist for convenience, but named stages are the maintainable choice.

`COPY INTO <table>` reads files from a stage into a table, and `COPY INTO <location>` unloads a query result out. The command is parallel: a warehouse loads many files at once, and file size drives how well that parallelism is used. `FILE_FORMAT` describes the data (CSV, JSON, Parquet, Avro, ORC, XML), and `MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE` lets a semi-structured load map fields to columns by name so schema drift does not break it.

Several options matter in production. `PATTERN` restricts which files a load considers, which is how you batch by date or prefix. `ON_ERROR` decides what happens to bad rows: `ABORT_STATEMENT` (default), `CONTINUE`, `SKIP_FILE`, or the variants that skip files based on an error threshold. `VALIDATION_MODE` runs a dry run that reports errors without loading. `PURGE = TRUE` deletes staged files after a successful load. `FORCE = FALSE` (the default) makes Snowflake skip files it has already loaded in the last 64 days, using load metadata.

The gotcha is file sizing and count. Snowflake recommends compressed files of roughly 100 to 250 MB; thousands of kilobyte files waste parallelism and load metadata, while a single multi-gigabyte file cannot be split across the warehouse effectively and becomes a single-threaded load. Watch `COPY_HISTORY` for `error_count` and `status`, and remember that unloading to a single file is similarly serial.

```sql
-- External stage over your own bucket, credential-free via a storage integration.
CREATE OR REPLACE STORAGE INTEGRATION s3_orders_int
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('s3://acme-raw/orders/');

CREATE OR REPLACE STAGE raw.orders.landing
  URL = 's3://acme-raw/orders/'
  STORAGE_INTEGRATION = s3_orders_int
  FILE_FORMAT = (TYPE = PARQUET);

-- Load one day of files, tolerate bad rows, and clean up after success.
COPY INTO raw.orders.orders
FROM @raw.orders.landing
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
PATTERN = '.*2026-10-04.*[.]parquet'
ON_ERROR = 'CONTINUE'
PURGE = TRUE
FORCE = FALSE;

-- Dry run: report errors without loading anything.
COPY INTO raw.orders.orders FROM @raw.orders.landing
  FILE_FORMAT = (TYPE = PARQUET)
  VALIDATION_MODE = 'RETURN_ERRORS';

-- Unload a result set, splitting it into many files.
COPY INTO @raw.orders.exports/daily/
FROM (SELECT * FROM analytics.marts.daily_revenue WHERE order_date = '2026-10-04')
FILE_FORMAT = (TYPE = PARQUET)
MAX_FILE_SIZE = 100000000
HEADER = TRUE;

-- Load history for the last day, including errors.
SELECT file_name, status, row_count, error_count, first_error_message, last_load_time
FROM TABLE(information_schema.copy_history(
  table_name => 'raw.orders.orders',
  start_time => DATEADD('day', -1, CURRENT_TIMESTAMP())))
ORDER BY last_load_time DESC;
```

Stages abstract storage locations and `COPY INTO` loads or unloads in parallel — feed it 100 to 250 MB compressed files, use `PATTERN`, `ON_ERROR`, and `VALIDATION_MODE` deliberately, and watch `COPY_HISTORY` for failures.

Load metadata is Snowflake's memory of which files it has already loaded into a table, and it lasts 64 days. `FORCE = FALSE` consults it and skips a file already loaded; `FORCE = TRUE` ignores it. The memory belongs to the table, so the same file loaded into a second table is loaded again.

`ON_ERROR` gives you a policy per statement: `ABORT_STATEMENT` fails the load, `CONTINUE` loads good rows and records bad ones, and `SKIP_FILE` skips a file containing an error. For Snowpipe the policy lives in the pipe definition, and every outcome lands in `COPY_HISTORY` with an `error_count` and a status, which is the first place to look when data never appears.

File size drives parallelism: compressed files of roughly 100 to 250 MB let the warehouse keep many files in flight at once, while one multi-gigabyte file limits how much of the warehouse can be used. Unloading has the same property, which is what `MAX_FILE_SIZE` is for.

## How do you handle semi-structured data with VARIANT?

`VARIANT` stores JSON, Avro, Parquet, ORC, or XML in a single column. Snowflake parses the value, extracts the keys it can see, and stores them in an internal columnar form with metadata about the paths present, so path lookups are fast even though the source is schema-on-read. It is the reason you can land raw events without flattening them first.

Access fields with `:` and cast with `::`. `payload:user.id::NUMBER` navigates into nested objects, and `TRY_CAST` is safer than `CAST` when values may be missing or malformed. Arrays are exploded with `LATERAL FLATTEN`, which turns each element into a row and exposes `VALUE`, `PATH`, and `INDEX`. `OBJECT_KEYS` and `TYPEOF` help when you are exploring an unfamiliar payload.

The recommended pattern is two layers. Keep the raw `VARIANT` exactly as it arrived, because that is replayable and tolerant of schema drift, then flatten the fields people actually query into typed columns in a view, materialized view, or dynamic table. Typed columns prune and join better, and they document the contract that downstream consumers rely on.

The gotcha is treating `VARIANT` as the schema. Filters and joins on `payload:some_field` are slower and less predictable than on a typed column, and a path that exists in only some rows produces nulls silently. Also avoid indexing deep paths in tight loops: extract once into a typed column or a view rather than casting the same expression repeatedly across a large scan.

```sql
CREATE OR REPLACE TABLE raw.events (event_id STRING, payload VARIANT);

COPY INTO raw.events (event_id, payload)
FROM (
  SELECT $1:event_id::STRING, $1
  FROM @raw.events.event_stage
)
FILE_FORMAT = (TYPE = JSON STRIP_OUTER_ARRAY = TRUE);

-- Explore before you flatten.
SELECT payload:event_type::STRING AS event_type, TYPEOF(payload:amount) AS amount_type,
       OBJECT_KEYS(payload) AS keys
FROM raw.events
LIMIT 5;

-- Query typed fields with a safe cast.
SELECT payload:user.id::NUMBER AS user_id,
       TRY_CAST(payload:amount AS NUMBER(12, 2)) AS amount
FROM raw.events
WHERE payload:event_type::STRING = 'purchase';

-- Explode an array into rows.
SELECT e.event_id, i.value:sku::STRING AS sku, i.value:qty::NUMBER AS qty
FROM raw.events e,
     LATERAL FLATTEN(input => e.payload:items) i;

-- Promote the fields people actually use into typed columns.
CREATE OR REPLACE VIEW analytics.v_events AS
SELECT event_id,
       payload:event_type::STRING AS event_type,
       payload:user.id::NUMBER AS user_id,
       TRY_CAST(payload:amount AS NUMBER(12, 2)) AS amount,
       payload:ts::TIMESTAMP_NTZ AS event_ts
FROM raw.events;
```

Keep raw `VARIANT` for replay and drift, use `:` with `TRY_CAST` and `FLATTEN` to explore it, and promote the fields you actually query into typed columns so pruning and joins behave predictably.

When a document is loaded, Snowflake parses it and writes each distinct path into an internal columnar representation, so a key repeated across millions of rows is stored once as a path with its values beside it rather than as text in every row. Statistics exist for variant paths, but they are weaker than on typed columns, which is why a predicate on `payload:user.id` prunes worse than one on a typed `user_id`.

Path syntax uses `:` for object keys and `[]` for array elements, and both accept a constant or a computed key. Keys containing spaces or special characters need double quotes or bracket syntax, for example `payload:"user id"` or `payload['user id']`. `FLATTEN` turns an array into rows and exposes `VALUE`, `PATH`, `INDEX`, and `THIS`.

`TRY_CAST` returns null both when a value is missing and when it is malformed, so a quiet null can hide a schema change. When a path is business-critical, validate it with `TYPEOF` or check `OBJECT_KEYS`, and promote it into a typed column instead of recasting it in every query.

## How do external tables work?

An external table is a read-only table whose data stays in your own cloud storage. Snowflake stores only the metadata and the partition information, and reads the Parquet, ORC, Avro, or JSON files at query time. The appeal is querying a data lake in place with no load step and no duplicate storage bill.

You define columns over the file structure, optionally deriving them from the file's value columns with `AS (VALUE:c1::STRING)`, and point the table at a stage with `LOCATION`. `AUTO_REFRESH = TRUE` lets Snowflake pick up new files automatically when the external stage has event notifications configured; otherwise you run `ALTER EXTERNAL TABLE ... REFRESH` to scan for new files. `METADATA$FILENAME` and `METADATA$FILE_ROW_NUMBER` are available for lineage and debugging.

Performance depends almost entirely on how the files are laid out in object storage. Snowflake can prune on partition-like path prefixes and on the statistics in Parquet footers, so a directory layout such as `event_date=2026-10-04/` makes date filters selective. External tables support no clustering, no Time Travel, no materialized views, and no DML.

The gotcha is treating an external table as a warehouse table. Repeatedly querying cold lake data through an external table pays object-storage latency on every scan and cannot benefit from many Snowflake optimizations. The common pattern is to keep cold or rarely queried history external and load hot, frequently joined data into native tables, or to use Iceberg tables when you need both openness and write support.

```sql
CREATE OR REPLACE EXTERNAL TABLE raw.lake.events_ext (
  event_id STRING AS (VALUE:c1::STRING),
  payload  VARIANT AS (VALUE)
)
LOCATION = @raw.lake.events_stage/events/
FILE_FORMAT = (TYPE = PARQUET)
AUTO_REFRESH = TRUE;

-- Manual refresh when auto-refresh is not configured.
ALTER EXTERNAL TABLE raw.lake.events_ext REFRESH;

-- Query in place, with file metadata for lineage.
SELECT METADATA$FILENAME, METADATA$FILE_ROW_NUMBER, event_id, payload:event_type::STRING
FROM raw.lake.events_ext
WHERE event_id IS NOT NULL
LIMIT 10;

-- Seen files and their partitions, useful when pruning looks wrong.
SELECT COUNT(*) AS files, MIN(METADATA$FILENAME) AS sample_file
FROM raw.lake.events_ext;
```

External tables query your lake in place with no load and no copy, but they are read-only and depend on file layout, so keep hot, joined data native and leave cold history external.

An external table holds no row data in Snowflake: only column definitions and a catalog of the files and partitions found in the stage. At query time the warehouse opens the underlying objects directly, using path-derived partition columns and the statistics in each file's footer to choose files. Pruning therefore depends on directory layout and on the source format's own statistics, not on Snowflake micro-partitions.

Because the files live outside Snowflake, features tied to storage internals do not apply: external tables are read-only, have no Time Travel, cannot be clustered, and cannot be written by DML. `AUTO_REFRESH = TRUE` picks up new files only when the stage has event notifications; otherwise the file list stays stale until `ALTER EXTERNAL TABLE ... REFRESH` runs.

A layout that puts every date in one directory removes the only cheap pruning signal. Naming prefixes by partition, such as `event_date=2026-10-04/`, is what lets a date filter avoid opening unrelated files.

## How do Snowflake Iceberg tables work?

An Iceberg table stores data as Parquet files plus Iceberg metadata, either in a Snowflake-managed external volume or in a catalog you already own. The open format is the point: Spark, Trino, Flink, and other engines can read and write the same table that Snowflake queries, so no engine is locked out and you stop copying data into a warehouse-native format for every consumer.

There are two ownership models. A Snowflake-managed Iceberg table (`CATALOG = 'SNOWFLAKE'`) is written through Snowflake, which manages the metadata and the external volume. A catalog-linked table (`CATALOG = 'glue_catalog_integration'` or an Iceberg REST catalog) is registered from an external catalog, and Snowflake treats it as read-mostly while another engine owns the writes. Snowflake can also write to externally managed tables in some configurations, but ownership must be explicit.

Snowflake-managed Iceberg tables support many native features, including `INSERT`, `MERGE`, Time Travel, and clustering, because Snowflake controls the metadata. Catalog-linked tables get whatever the external catalog provides and generally cannot use Snowflake-only features such as automatic clustering or Snowflake-managed snapshots.

The gotchas are governance and maintenance. You must create an external volume and a catalog integration, decide who may write, and keep the metadata maintained. Cross-engine writers can conflict, so designate one writer per table. And because the data lives in your storage, storage cost, lifecycle policies, and file compaction become your responsibility rather than Snowflake's.

```sql
CREATE OR REPLACE EXTERNAL VOLUME iceberg_vol
  STORAGE_LOCATIONS = ((
    NAME = 's3_primary'
    STORAGE_PROVIDER = 'S3'
    STORAGE_BASE_URL = 's3://acme-iceberg/'
    STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::111122223333:role/snowflake-iceberg'
  ));

-- Snowflake-managed: Snowflake writes and maintains the Iceberg metadata.
CREATE OR REPLACE ICEBERG TABLE analytics.orders_iceberg (
  order_id NUMBER,
  order_date DATE,
  amount NUMBER(12, 2)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'iceberg_vol'
BASE_LOCATION = 'analytics/orders';

-- Catalog-linked: an external catalog owns the table.
CREATE OR REPLACE ICEBERG TABLE analytics.orders_glue
  CATALOG = 'glue_catalog_integration'
  CATALOG_TABLE_NAME = 'orders'
  EXTERNAL_VOLUME = 'iceberg_vol';

INSERT INTO analytics.orders_iceberg
SELECT order_id, order_date, amount FROM raw.orders_staging;

SELECT COUNT(*) AS rows, COUNT(DISTINCT order_date) AS days
FROM analytics.orders_iceberg;
```

Iceberg tables put Snowflake on the same open Parquet-and-metadata table as Spark and Trino, so choose deliberately between a Snowflake-managed table and a catalog-linked one, and make write ownership explicit.

An Iceberg table is a set of Parquet data files plus a metadata tree: manifests list data files, a manifest list lists manifests, and a snapshot points at one manifest list. A writer commits by adding a new snapshot atomically, so two engines can share the table without coordinating on files, as long as only one writer owns a given table at a time.

Snowflake needs an external volume to reach the storage layer even for a catalog-linked table. A Snowflake-managed table (`CATALOG = 'SNOWFLAKE'`) is written through Snowflake, which maintains the metadata and can apply Snowflake features. A catalog-linked table is registered from an external catalog that owns the metadata, and Snowflake is generally a reader there.

Openness moves responsibility to you. File compaction, snapshot expiration, and storage lifecycle are no longer handled by a warehouse-native format, and cross-engine writers can commit conflicting snapshots, so designate a single writer per table.

## What is Snowpark and how do you use Python in Snowflake?

Snowpark is the developer framework that lets you write DataFrame-style code in Python, Java, or Scala that is translated into SQL and executed inside Snowflake. Your client builds a lazy plan; Snowflake runs it near the data, so you do not pull millions of rows into a local process and you do not need a separate Spark cluster for data that already lives in Snowflake.

The programming model mirrors pandas or PySpark enough to be familiar: `session.table()` returns a DataFrame, `filter`, `select`, `join`, `group_by`, and `agg` build up a plan, and an action such as `collect()`, `count()`, `show()`, or `save_as_table()` triggers execution. Because the plan is lazy, an error in a transformation may not surface until the action, and pushing work into Snowflake expressions rather than Python loops is what keeps it fast.

Snowpark also hosts user-defined functions and stored procedures written in Python, and it supports `session.sql()` when you prefer raw SQL. Memory-heavy work such as large UDFs, pandas-on-Snowpark, or in-memory model scoring benefits from a Snowpark-optimized warehouse, which has more memory per node than a standard warehouse.

The gotchas are subtle. A Python UDF runs once per row and is far slower than an equivalent vectorized SQL expression, so use native functions where they exist. Pulling results with `collect()` into the client defeats the purpose for large data. And a Snowpark-optimized warehouse costs more per credit, so reserve it for genuinely memory-bound workloads.

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum as sum_

session = Session.builder.configs(
    {"account": "acme-org_acme", "user": "TRANSFORMER", "password": "<password>",
     "role": "TRANSFORM_ROLE", "warehouse": "TRANSFORM_WH"}
).create()

orders = session.table("RAW.ORDERS.ORDERS")
daily = orders.group_by(col("ORDER_DATE")).agg(sum_(col("AMOUNT")).alias("REVENUE"))
daily.write.mode("overwrite").save_as_table("ANALYTICS.MARTS.DAILY_REVENUE")

```python
# The plan is lazy: nothing runs until an action, and collect() should stay small.
filtered = orders.filter(col("ORDER_DATE") >= "2026-10-01")
print(filtered.count())                 # action, returns a single number
filtered.show(10)                       # action, prints ten rows

# session.sql() is available when raw SQL is clearer than DataFrame chains.
session.sql("ALTER SESSION SET QUERY_TAG = 'snowpark:daily_revenue'").collect()
```

Snowpark lets you express transformations as a lazy DataFrame plan that Snowflake executes next to the data, so keep work server-side, prefer native functions over row-by-row UDFs, and use a Snowpark-optimized warehouse only for memory-bound jobs.

Snowpark does not run Python on the warehouse for a DataFrame expression. The client builds a logical plan, which cloud services compiles to SQL and hands to the warehouse; only code inside a user-defined function or a stored procedure actually executes in a Python runtime. That is why DataFrame transforms perform like the SQL they compile to, while a Python UDF over the same rows does not.

UDFs come in two shapes. A scalar Python UDF is invoked once per row and pays interpreter overhead on every call. A vectorized pandas UDF receives batches as pandas Series and returns a Series, amortizing that overhead across many rows; reach for it when the logic cannot be expressed in native SQL.

A Snowpark-optimized warehouse has more memory and local SSD per node for memory-heavy UDFs and model scoring, and it costs more per credit than a standard warehouse. Use it when a job genuinely spills or needs the memory, not as a default.

## What are UDFs and stored procedures?

A user-defined function returns a value per row and is called from SQL, so it composes with `SELECT`, `WHERE`, and joins. UDFs can be written in SQL, JavaScript, Python, or Java, and can be scalar (one value per row) or return a table. A stored procedure runs procedural logic, may execute DML and DDL, and is invoked with `CALL`. The distinction matters: UDFs extend expressions, procedures run workflows.

UDFs are pure by design. A scalar UDF cannot query other tables, which keeps it deterministic and parallelizable, and that is exactly why it is safe to use inside a large scan. Procedures, by contrast, can run multiple statements, use Snowflake Scripting control flow, and coordinate a multi-step load or transformation. Procedures can be called from tasks and execute as a chosen role when you use `EXECUTE AS` semantics.

Deployment follows the language. SQL and JavaScript UDFs are `CREATE FUNCTION` statements; Python and Java UDFs are created with a handler, a runtime version, and packages, and are often deployed from Snowpark with `@udf` and `stage_location`. Permanent functions need a stage for their code, and callers need `USAGE` on the function and the database and schema that hold it.

In [ ]:
from snowflake.snowpark.functions import udf


@udf(name="mask_email", replace=True, is_permanent=True, stage_location="@utils.udf_stage")
def mask_email(value: str) -> str:
    if not value or "@" not in value:
        return "invalid"
    return value[0] + "***" + value[value.index("@"):]

```sql
-- A SQL scalar UDF: pure, composable, and safe inside a large scan.
CREATE OR REPLACE FUNCTION analytics.fiscal_quarter(d DATE)
  RETURNS STRING
  LANGUAGE SQL
AS
$$
  CASE WHEN MONTH(d) >= 2 THEN 'Q' || CEIL((MONTH(d) - 1) / 3.0)::INT
       ELSE 'Q4' END
$$;

SELECT analytics.fiscal_quarter(order_date) AS fq, SUM(amount)
FROM analytics.orders
GROUP BY fq;

-- A Snowflake Scripting procedure: multiple statements and control flow.
CREATE OR REPLACE PROCEDURE analytics.load_daily(load_date DATE)
  RETURNS STRING
  LANGUAGE SQL
AS
$$
BEGIN
  DELETE FROM analytics.marts.daily_revenue WHERE order_date = :load_date;
  INSERT INTO analytics.marts.daily_revenue
  SELECT order_date, SUM(amount)
  FROM raw.orders.orders
  WHERE order_date = :load_date
  GROUP BY order_date;
  RETURN 'loaded ' || :load_date::STRING;
END;
$$;

CALL analytics.load_daily('2026-10-04');
```

UDFs extend SQL with a value per row and cannot query tables, while stored procedures orchestrate multi-statement work — keep UDFs pure and small, and remember that a row-by-row Python UDF is far slower than a native SQL expression.

A scalar UDF must be side-effect free and cannot query tables, which lets the optimizer inline or parallelize it across a scan. A stored procedure is different: it is invoked with `CALL`, runs its statements in order with the privileges of its owner role by default, and can contain DML, DDL, and control flow. A procedure's statements are not one automatic transaction; only what you wrap in an explicit transaction commits or rolls back together.

Python UDFs and procedures are packaged with a runtime version, a handler naming the callable, optional `PACKAGES`, and optional `IMPORTS` for staged files. Snowflake stages the code and loads it into the execution environment, so the artifact must be uploaded before `CREATE FUNCTION` and re-uploaded when the code changes.

A procedure is the right place to coordinate steps, not to process rows. Looping over rows in a procedure multiplies statement overhead and defeats set-based execution; keep the loop inside one statement, or batch the work so each call handles many rows.

## What is the RBAC role and privilege model?

Access control in Snowflake is role-based. Privileges are granted to roles, roles are granted to other roles to form a hierarchy, and users are granted roles. Users never receive privileges on objects directly in a well-designed setup; they activate a role and inherit everything that role can do, which makes access auditable and revocable in one place. The built-in roles are `ACCOUNTADMIN`, `SYSADMIN`, `SECURITYADMIN`, `USERADMIN`, and `PUBLIC`.

Roles come in two useful shapes. Access roles grant privileges on databases, schemas, tables, and warehouses, and are usually functional, one per job to be done such as `analytics_engineer` or `bi_reader`. Functional or business roles are granted access roles and then granted to users. The `SYSADMIN` role owns objects, and best practice is to build the hierarchy so `SYSADMIN` is granted every custom role that owns objects, keeping `ACCOUNTADMIN` for account-level administration only.

Privilege grants have breadth as well as depth. `GRANT SELECT ON ALL TABLES IN SCHEMA` covers existing tables, but new tables will not be visible unless you also grant on `FUTURE TABLES`. The same future-grant pattern applies to schemas, views, and stages, and it is the mechanism that makes automated deployments work without a human re-granting after every release.

The gotchas are `OWNERSHIP` and the built-in roles. `OWNERSHIP` implies every other privilege on an object and cannot be granted partially, so it should be held deliberately and usually by a role you manage centrally. `ACCOUNTADMIN` can bypass many controls and should never be the role an application, a scheduled task, or an analyst session runs as. Grant the minimum, prefer functional roles, and audit grants with `GRANTS_TO_ROLES` or `SHOW GRANTS`.

```sql
USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS analytics_engineer;

USE ROLE SYSADMIN;
GRANT USAGE ON WAREHOUSE transform_wh TO ROLE analytics_engineer;
GRANT USAGE ON DATABASE analytics TO ROLE analytics_engineer;
GRANT USAGE ON SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT SELECT ON FUTURE TABLES IN SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT ROLE analytics_engineer TO ROLE SYSADMIN;

-- Give a person the role, not the privileges.
GRANT ROLE analytics_engineer TO USER analyst_jane;

-- Audit who has what.
SHOW GRANTS TO ROLE analytics_engineer;
SELECT * FROM snowflake.account_usage.grants_to_roles
WHERE role LIKE 'ANALYTICS%' AND deleted_on IS NULL;
```

Grant privileges to functional roles, grant roles to roles and users, cover new objects with future grants, and keep `ACCOUNTADMIN` and `OWNERSHIP` deliberate rather than routine.

The predefined roles divide responsibility. `ACCOUNTADMIN` holds account-level settings and can see everything; `SYSADMIN` owns warehouses and databases; `SECURITYADMIN` manages grants and can monitor any object; `USERADMIN` creates users and roles; `PUBLIC` is granted to every user automatically. `SYSADMIN` and `SECURITYADMIN` are granted to `ACCOUNTADMIN`, which is why `ACCOUNTADMIN` inherits broad access rather than being granted it object by object.

A future grant is stored on the schema and applied to objects created later; it is not retroactive and does not reach objects that already exist. A deployment that creates a new table still needs the future grant to have been issued earlier, or an explicit grant afterward. Ownership is separate from privileges: exactly one role owns an object, and `OWNERSHIP` implies every privilege on it.

Because `ACCOUNTADMIN` inherits `SYSADMIN`, running applications or scheduled tasks as `ACCOUNTADMIN` makes least-privilege auditing impossible. Transfer ownership to custom roles, grant those roles to `SYSADMIN`, and keep `ACCOUNTADMIN` for account administration only.

## What are secure views, row access policies, and column masking?

These three features enforce least privilege below the SQL layer, so a rule applies no matter which tool issues the query. A secure view hides its definition from users and disables optimizer shortcuts that could otherwise leak information about the underlying data. A row access policy is a function attached to a table or view that returns true or false per row, deciding which rows a role may see. A masking policy is a function attached to a column that rewrites the value returned to a role.

Row access policies are the right tool for multi-tenant or regional isolation. You attach the policy to the column it filters, and the policy body typically compares `CURRENT_ROLE()` with a mapping table. Because the filter is applied by the engine, every consumer — BI tool, notebook, ad-hoc SQL — is constrained, and no dashboard can forget to add the tenant predicate. Masking policies handle the column case: analysts see a redacted email or a nulled national id, while a privileged role sees the real value.

Secure views matter for a different reason. A normal view's definition is visible in the object metadata, and the optimizer may use internal information to plan a query in ways that expose intermediate data. Marking a view `SECURE` hides the SQL and forces a plan that does not leak, which makes it the right wrapper when a view encodes a compliance rule rather than just a convenience join.

The gotchas are edition and correctness. Row access policies and masking policies require Enterprise Edition, so a Standard Edition account cannot use them at all. Both policies should key on `CURRENT_ROLE()` rather than `CURRENT_USER()`, because users change and roles are the stable unit of governance. A policy that returns null can fail open or closed depending on context, and a policy that queries a mapping table on every row can slow large scans, so keep policy logic simple.

```sql
CREATE OR REPLACE SECURE VIEW analytics.v_orders AS
SELECT order_id, customer_id, amount
FROM analytics.orders
WHERE order_status <> 'test';

CREATE OR REPLACE ROW ACCESS POLICY security.region_policy
  AS (region STRING) RETURNS BOOLEAN ->
    CURRENT_ROLE() = 'ACCOUNTADMIN'
    OR EXISTS (
      SELECT 1 FROM security.region_map m
      WHERE m.role_name = CURRENT_ROLE() AND m.region = region
    );
ALTER TABLE analytics.orders
  ADD ROW ACCESS POLICY security.region_policy ON (region);

CREATE OR REPLACE MASKING POLICY security.email_mask
  AS (val STRING) RETURNS STRING ->
    CASE WHEN CURRENT_ROLE() IN ('PII_ADMIN', 'ACCOUNTADMIN') THEN val
         ELSE REGEXP_REPLACE(val, '^[^@]+', '***') END;
ALTER TABLE analytics.customers
  MODIFY COLUMN email SET MASKING POLICY security.email_mask;

-- See which policies protect which objects.
SHOW ROW ACCESS POLICIES;
SHOW MASKING POLICIES;
```

Secure views, row access policies, and masking policies enforce least privilege in the engine rather than in each query, keyed on `CURRENT_ROLE()` — and row and column policies require Enterprise Edition.

Row access and masking policies are evaluated at query time against the role active in the session, not stored per user or per tool. The same SQL run by two roles can return different rows or different column values, which is why enforcement survives a client that forgets to add a filter. Because a policy is a function, it must be deterministic enough for the engine to plan and is called as part of the scan.

Order matters conceptually. The row access policy decides which rows exist for the role, and the masking policy decides what the returned values look like for the rows that survive. A masking policy can use `CURRENT_ROLE()` and `CURRENT_USER()` to choose who sees the real value, while a row access policy usually compares the role against a mapping table. `CURRENT_ROLE()` is the safer key, because roles are the unit of governance and users change.

Both features require Enterprise Edition. A policy that calls a mapping table adds work to every query that touches the protected object, so keep the mapping small and the logic simple. A policy that returns null instead of false can behave differently from a false, so be explicit with boolean expressions.

## How do resource monitors control cost?

A resource monitor tracks credit consumption for the account or for a set of warehouses against a quota and fires actions at thresholds. It is the enforcement mechanism for cost control: without one, credits grow silently; with one, the account can notify, suspend, or hard-stop a warehouse when spend crosses a line. A monitor has a `CREDIT_QUOTA`, a `FREQUENCY` that resets the counter (monthly, daily, weekly, or never), and a list of triggers at percentages.

Triggers have three levels of severity. `DO NOTIFY` sends an alert, typically to account administrators, which is the right action early on. `DO SUSPEND` lets statements that are already running finish and then suspends the warehouse, which avoids corrupting an in-flight load. `DO SUSPEND_IMMEDIATE` cancels running statements and suspends at once, which is the emergency brake. Notify early and escalate slowly: a 75 percent notification, a 100 percent suspend, and a 110 percent immediate suspend is a common shape.

Monitor granularity matters. A monitor attached to the account covers any warehouse without its own monitor; a monitor attached to a warehouse governs just that warehouse. In practice you set a generous account-level guardrail and tighter per-warehouse quotas for the biggest consumers, so one runaway job cannot take the whole account down. Serverless features such as Snowpipe, automatic clustering, and materialized-view maintenance also consume credits, and account-level monitors include them.

The gotcha is that a resource monitor is a backstop, not an optimization. `DO SUSPEND` stops compute but does not fix an expensive query, and `SUSPEND_IMMEDIATE` will kill a legitimate long-running load. Pair monitors with observability: `WAREHOUSE_METERING_HISTORY` shows actual credits, and `QUERY_ATTRIBUTION_HISTORY` attributes them to queries and tags. Set the quota from measured usage, not from a guess, and revisit it as workloads change.

```sql
CREATE OR REPLACE RESOURCE MONITOR elt_monitor WITH CREDIT_QUOTA = 2000
  FREQUENCY = MONTHLY
  TRIGGERS
    ON 75 PERCENT DO NOTIFY
    ON 100 PERCENT DO SUSPEND
    ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE elt_wh SET RESOURCE_MONITOR = elt_monitor;

-- Account-level guardrail for warehouses without their own monitor.
CREATE OR REPLACE RESOURCE MONITOR account_monitor WITH CREDIT_QUOTA = 20000
  FREQUENCY = MONTHLY
  TRIGGERS ON 90 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
ALTER ACCOUNT SET RESOURCE_MONITOR = account_monitor;

-- Credits actually consumed, per warehouse and day.
SELECT warehouse_name, DATE_TRUNC('day', start_time) AS usage_day,
       SUM(credits_used) AS credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -14, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY usage_day DESC, credits DESC;

SHOW RESOURCE MONITORS;
```

A resource monitor turns cost from an observation into an enforced limit — notify early, suspend before an emergency, and set quotas and per-warehouse monitors from measured usage rather than guesswork.

A monitor totals the credits used by its assigned warehouses plus the cloud services that supported them, within an interval that resets at 12:00 AM UTC. Only `ACCOUNTADMIN` can create one, a warehouse can belong to at most one warehouse monitor, and an account monitor does not override the assignment of individual warehouses: either monitor can trigger a suspend.

The important exemptions are serverless features and cloud services. An account-level resource monitor does not control Snowflake-managed compute for Snowpipe, automatic clustering, or materialized views, and budgets exist for those. A warehouse-level monitor can observe cloud-services credits but cannot suspend them, and a suspension clears when the interval resets, the quota is raised, or the warehouse is unassigned.

Monitor thresholds are calculated on all consumed cloud-services credits, ignoring the 10 percent daily adjustment, so a monitor can report 100 percent of quota while the actual bill is smaller. If you need a hard guarantee, set the suspend threshold below quota, for example 90 percent, because suspension takes effect slightly after the threshold is crossed.

## How do transactions and isolation work in Snowflake?

Snowflake is ACID-compliant across all supported operations. The default isolation level is read-committed snapshot isolation: each statement sees a consistent snapshot of the data as of the moment the statement began, and uncommitted changes from other transactions are invisible. Because micro-partitions are immutable, a reader simply addresses the versions that existed at its snapshot, which is what lets readers run without blocking writers.

An explicit transaction starts with `BEGIN` (or `BEGIN TRANSACTION`), spans one or more DML statements, and ends with `COMMIT` or `ROLLBACK`. `SERIALIZABLE` isolation is available per session or per transaction with `ALTER SESSION SET TRANSACTION_ISOLATION_LEVEL = 'SERIALIZABLE'`, which detects and aborts conflicting concurrent writes rather than allowing them to interleave. It costs more contention, so use it where correctness demands it, such as balance updates.

DDL is the important exception. Statements such as `CREATE`, `ALTER`, and `DROP` run in their own transaction and auto-commit regardless of any surrounding explicit transaction, so a `CREATE` followed by a failure and a `ROLLBACK` does not undo the create. Wrap a DDL plus its dependent DML in a stored procedure if you need all-or-nothing behavior, and be careful about mixing DDL into long transactions.

The gotcha is where contention appears. Snowflake does not expose traditional lock waits; instead, concurrent writers to the same rows are serialized by the transaction manager, and readers never block. What you see is not a lock wait but queueing and longer execution times. Long-running transactions also hold historical versions longer, which increases storage and can affect Time Travel windows.

```sql
ALTER SESSION SET TRANSACTION_ISOLATION_LEVEL = 'SERIALIZABLE';

BEGIN;
UPDATE analytics.accounts SET balance = balance - 100 WHERE account_id = 1;
UPDATE analytics.accounts SET balance = balance + 100 WHERE account_id = 2;
COMMIT;

-- DDL auto-commits independently: rolling back does not undo the CREATE.
BEGIN;
CREATE TABLE analytics.tmp_check (id NUMBER);
ROLLBACK;
SHOW TABLES LIKE 'TMP_CHECK' IN SCHEMA analytics;
```

Snowflake gives ACID transactions with read-committed snapshot isolation and non-blocking readers, and SERIALIZABLE when you need it — just remember that DDL auto-commits outside your transaction.

The default isolation is read committed at the statement level: each statement takes a fresh snapshot when it begins, so two statements in the same explicit transaction can see different versions if a commit lands between them. Readers do not block writers and writers do not block readers, because a reader simply addresses the versions in its snapshot. Concurrent DML against the same table is serialized through the transaction manager rather than through a visible lock wait.

Outside an explicit transaction every statement is its own transaction and commits on success; that is autocommit. `BEGIN` opens a transaction that stays open until `COMMIT` or `ROLLBACK`, and DDL inside it still commits immediately and independently. `SERIALIZABLE` changes conflict handling: instead of letting statements interleave, Snowflake aborts one of the conflicting transactions, so the application must be prepared to retry.

A long open transaction keeps the versions it can still read alive, which delays storage cleanup and can consume the Time Travel window. Keep transactions short and put long computation outside them; wrap only the statements that must be atomic.

## What is Snowflake data sharing?

Data sharing exposes read-only databases to other Snowflake accounts without copying data. The provider creates a share, grants privileges on the objects to share, and adds consumer accounts. The consumer creates a database from the share and queries it with their own warehouse. The provider's storage and governance stay in place, and the consumer sees new data as the provider loads it.

Because shares are metadata grants rather than exports, there is no ETL, no sync job, and no duplicate storage bill. The consumer pays only for the compute that queries the shared data. Shares are live: when the provider commits new rows, consumers see them on their next query, subject to the provider's Time Travel and privilege choices. This makes sharing a natural fit for vendor data products, internal cross-business-unit analytics, and partner data exchange.

Two extensions matter. Account-to-account sharing works within a region; to share across regions or cloud providers you replicate the share's database to a secondary account first and then share from there. Reader accounts let you share with an organization that does not have its own Snowflake account: the provider creates and manages the reader account and typically bills for its compute.

The gotchas are governance and privileges. Shares are read-only, so a consumer cannot write back; the provider controls what is exposed and must avoid leaking more than intended through an over-broad grant. Consumers need `IMPORTED PRIVILEGES` on the shared database, and a share whose database is replaced or dropped will break consumers unless the change is coordinated. Treat a share as a product with a contract, not as a one-off grant.

```sql
-- Provider side: grant privileges to the share, then add consumers.
CREATE SHARE sales_share;
GRANT USAGE ON DATABASE analytics TO SHARE sales_share;
GRANT USAGE ON SCHEMA analytics.marts TO SHARE sales_share;
GRANT SELECT ON TABLE analytics.marts.fct_orders TO SHARE sales_share;
ALTER SHARE sales_share ADD ACCOUNTS = consumer_org.consumer_acct;

-- Consumer side: mount the share and use it.
CREATE DATABASE shared_sales FROM SHARE provider_org.provider_acct.sales_share;
GRANT IMPORTED PRIVILEGES ON DATABASE shared_sales TO ROLE analyst;

SELECT COUNT(*) FROM shared_sales.marts.fct_orders;

-- Cross-region sharing: replicate first, then share from the secondary.
ALTER DATABASE analytics ENABLE REPLICATION TO ACCOUNTS secondary_org.secondary_acct;
CREATE DATABASE analytics_replica FROM SHARE secondary_org.secondary_acct.sales_share;
```

Data sharing gives consumers live, read-only access to your tables with no copy and no ETL, so govern the grants like a product and replicate when you need to cross a region or cloud.

A share is a named set of grants on objects in a database. When a consumer creates a database from the share, they receive a read-only mount that resolves to the provider's storage and metadata; no rows are copied at share time and no synchronization job exists. New rows committed by the provider are visible to the consumer on their next query, bounded by the provider's grants and any Time Travel the provider applies.

Consumers need `IMPORTED PRIVILEGES` on the mounted database and pay for the compute that reads it. Cross-region or cross-cloud sharing requires replicating the database first, because a share is region-scoped, and replication is serverless and billed separately. Reader accounts exist for consumers without their own Snowflake account and are created and managed by the provider.

A share can expose object definitions as well as data, so treat it as a product with a schema contract. Replacing or dropping the shared database breaks every consumer; coordinate changes and prefer additive schema evolution.

## What is the search optimization service?

Search optimization adds a persistent, index-like structure to a table that makes highly selective point lookups fast. It targets exactly the predicates that micro-partition pruning handles poorly: equality and `IN` on a column, leading-wildcard `LIKE` such as `'%foo'`, `ARRAY_CONTAINS`, and geospatial predicates. Without it, those queries scan every micro-partition because min/max ranges cannot narrow them.

You enable it per table with `ALTER TABLE ... ADD SEARCH OPTIMIZATION`, optionally restricted to a set of columns or to specific methods such as `EQUALITY`, `SUBSTRING`, or `GEO`. The service builds and maintains the structure as the table changes, using serverless compute, and the optimizer uses it automatically when a query can benefit. Queries do not need to be rewritten to use it.

It pays off on large, relatively slowly changing tables where users run selective lookups, such as finding a customer by email, an order by id, or rows containing a substring. It also helps `LIKE '%pattern%'`-style searches that are otherwise full scans.

The gotcha is cost and churn. Maintenance runs on every DML, so a table with heavy insert and update volume pays continuously, and small or high-churn tables rarely benefit enough to justify it. Search optimization is also not a general accelerator: it does not speed up aggregations, joins, or broad range scans, and it does not replace pruning or clustering. Add it deliberately to a specific table, measure, and drop it if the queries it serves disappear.

```sql
-- Enable on selected columns and methods, not the whole table.
ALTER TABLE analytics.customers
  ADD SEARCH OPTIMIZATION ON EQUALITY(email, customer_id);
ALTER TABLE analytics.events
  ADD SEARCH OPTIMIZATION ON SUBSTRING(url);

-- These now use the optimized structure instead of a full scan.
SELECT customer_id, email FROM analytics.customers WHERE email = 'someone@example.com';
SELECT event_id FROM analytics.events WHERE url LIKE '%/checkout/complete%';

-- Inspect the configured optimization, then remove it if it stops paying.
SHOW SEARCH OPTIMIZATION ON analytics.customers;
ALTER TABLE analytics.customers DROP SEARCH OPTIMIZATION ON EQUALITY(email);
```

Search optimization makes selective point lookups and leading-wildcard searches fast where pruning cannot help, but it is maintained on every DML — enable it on large, slowly changing tables and measure against its serverless cost.

Search optimization maintains a persistent access path alongside the table for specific columns and predicate methods. The optimizer rewrites a matching predicate to use that structure instead of scanning micro-partitions, so the query keeps its original text. Maintenance is asynchronous: after DML the structure catches up in the background, so improvements are not instantaneous.

It targets predicates that min/max pruning cannot narrow: equality and `IN` on a column, a `LIKE` with a leading wildcard, `ARRAY_CONTAINS`, and geospatial predicates. It does not speed up aggregations, joins, or broad range scans, and enabling it changes no results; it only gives the optimizer another access path.

Cost is proportional to change volume, because every DML adds maintenance work, and on a high-churn staging table the maintenance credits can exceed anything saved. Enable it on large, slowly changing lookup tables, verify in the query profile that the access path is used, and remove it when those queries disappear.

## What are materialized views?

A materialized view stores the precomputed result of a query and is maintained automatically as the base table changes. The optimizer can transparently rewrite an eligible query to read the materialized view instead of recomputing from the base table, so applications keep writing ordinary SQL. It is the closest thing Snowflake has to a managed, refresh-on-write aggregate table.

Materialized views are deliberately narrow. In Snowflake a materialized view is defined over a single table, cannot contain joins, `HAVING`, window functions, or user-defined functions, and supports a limited set of aggregate and scalar expressions. That constraint is what makes incremental maintenance possible: the engine knows exactly which partitions changed and can update the stored aggregate without recomputing everything.

Use them for a small number of expensive aggregations on a moderately changing table, where many queries repeatedly ask the same grouped question. Because the optimizer can rewrite queries, you do not have to change application SQL, and maintenance runs on serverless compute rather than on your warehouse.

The gotchas are cost and staleness behavior. Maintenance consumes credits on every change to the base table, so a high-churn table can spend more on upkeep than it saves. Materialized views require Enterprise Edition, they are read-only in the sense that you cannot insert into them, and if they cannot be maintained they become stale rather than silently wrong. Always compare maintenance credits in `MATERIALIZED_VIEW_REFRESH_HISTORY` against the query time saved.

```sql
CREATE OR REPLACE MATERIALIZED VIEW analytics.mv_daily_revenue AS
SELECT order_date, SUM(amount) AS revenue, COUNT(*) AS orders
FROM analytics.orders
GROUP BY order_date;

-- The optimizer can rewrite this to read the materialized view.
SELECT order_date, SUM(amount) FROM analytics.orders GROUP BY order_date;

SHOW MATERIALIZED VIEWS IN SCHEMA analytics;

-- Pause maintenance (it may become stale), then resume.
ALTER MATERIALIZED VIEW analytics.mv_daily_revenue SUSPEND;
ALTER MATERIALIZED VIEW analytics.mv_daily_revenue RESUME;

-- Serverless credits spent maintaining it.
SELECT name, SUM(credits_used) AS credits
FROM snowflake.account_usage.materialized_view_refresh_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY name
ORDER BY credits DESC;
```

Materialized views are managed, incrementally maintained single-table aggregates that the optimizer can substitute into queries, so reserve them for repeated expensive aggregations and watch maintenance credits.

A materialized view stores its result as a Snowflake-managed internal table and is maintained from the base table's change tracking. Because maintenance must be incremental, Snowflake restricts the definition to a single base table with no joins, no `HAVING`, no window functions, and no user-defined functions, and only a subset of aggregates and scalar expressions is allowed.

The optimizer can transparently rewrite a query against the base table to read the materialized view when it can prove the result is equivalent, which is why application SQL does not change. Maintenance runs on serverless compute, and `ALTER MATERIALIZED VIEW ... SUSPEND` stops updates, after which the view may become stale until it is resumed.

A rewrite is not guaranteed. Predicates or expressions the optimizer cannot map to the view's grouping, and joins against the base table, may prevent substitution even when a human can see the equivalence. Confirm the rewrite in the query profile and watch maintenance credits, because a high-churn base table can cost more to maintain than the view saves.

## What is the query acceleration service?

The query acceleration service offloads parts of a scan and aggregation to serverless compute. When a query is eligible, Snowflake can process some of the table scan and some aggregation work on dynamically provisioned compute while the warehouse handles the rest, which shortens the query without resizing the warehouse.

It is most valuable when a warehouse is sized for the typical query but occasionally receives a much larger outlier. For example, a MEDIUM warehouse serving dashboards runs mostly small queries, then a month-end report scans ten times more data. Instead of scaling the whole warehouse to XL, you enable acceleration and let the service absorb the outlier. You enable it per warehouse with `ENABLE_QUERY_ACCELERATION = TRUE` and bound the fan-out with `QUERY_ACCELERATION_MAX_SCALE_FACTOR`.

Eligibility is decided by Snowflake. Queries that are good candidates tend to be scans with filters and aggregations over large tables; queries dominated by joins, sorts, or very selective point lookups are not. `SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>')` estimates whether a specific query would benefit, and `QUERY_HISTORY` exposes `QUERY_ACCELERATION_BYTES_SCANNED` so you can see what was actually offloaded.

The gotcha is that acceleration is metered and does not fix structure. It cannot rescue poor pruning, an exploding join, or a bad clustering choice, and the extra compute is billed separately from the warehouse. Use `SYSTEM$ESTIMATE_QUERY_ACCELERATION` before enabling it broadly, and compare the credits spent on acceleration against the credits saved by not resizing.

```sql
ALTER WAREHOUSE bi_wh SET
  ENABLE_QUERY_ACCELERATION = TRUE
  QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;

-- Would this specific query benefit?
SELECT SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>');

-- Which queries actually used acceleration, and how much did they scan?
SELECT query_id, warehouse_name,
       query_acceleration_bytes_scanned,
       query_acceleration_time,
       total_elapsed_time / 1000 AS elapsed_s
FROM snowflake.account_usage.query_history
WHERE query_acceleration_bytes_scanned > 0
  AND start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
ORDER BY query_acceleration_bytes_scanned DESC
LIMIT 20;

SHOW WAREHOUSES LIKE 'BI_WH';
```

Query acceleration absorbs occasional large scans with serverless compute so you do not have to size a warehouse for its worst query, but it is metered and never a substitute for good pruning and query shape.

For an eligible query the service provisions serverless compute that processes part of the scan and part of the aggregation while the warehouse handles the rest. Eligibility is decided at compile time from the query shape: scans with filters and aggregations over large tables qualify, while plans dominated by joins, sorts, or very selective lookups generally do not.

`QUERY_ACCELERATION_HISTORY` records the credits the service consumed, and `QUERY_HISTORY` exposes `QUERY_ACCELERATION_BYTES_SCANNED` and `QUERY_ACCELERATION_TIME` per query. `QUERY_ACCELERATION_MAX_SCALE_FACTOR` caps how much serverless compute can be added, which is the main guardrail on spend.

Acceleration is billed separately from the warehouse, so it can make a query faster and more expensive at the same time. It also cannot compensate for a full scan that pruning could have avoided; fix the predicate first.

## How do you migrate from Redshift, Hadoop, or another warehouse?

The migration pattern is consistent across sources. Land the source data as files in cloud storage, load it into Snowflake, translate the SQL and the semantics, validate the results, and only then cut over. Moving data before logic keeps the two systems comparable for as long as you need, and it decouples the mechanical data move from the more delicate re-expression of business logic.

Syntax translation is the part tooling handles best. SnowConvert and the Snowflake Migration Assistant translate Redshift, T-SQL, Oracle, Teradata, and Hive SQL toward Snowflake SQL, flagging constructs that have no direct equivalent. What they cannot decide for you is the data model, the permissions model, and the workload placement: Redshift distribution and sort keys become a combination of natural load order, optional clustering keys, and warehouse sizing; Hadoop partitioning becomes file layout and clustering.

Semantics are where migrations quietly break. Redshift and many engines treat empty strings and nulls differently, integer division differs, timestamps and time zones differ, and `MERGE` behavior and transaction isolation differ. Hive-style tables often carry partition columns in the path rather than the data, and loose typing in semi-structured sources can produce casts that silently truncate in Snowflake. Treat these as test cases rather than as documentation reading.

The gotchas are validation depth and cutover timing. Row counts are necessary but not sufficient: compare sums, distinct keys, and a sample of business reports against the source, and run both systems in parallel through at least one full reporting cycle. Keep the source running until reconciliation passes, and migrate in slices so a problem in one subject area does not block the rest. The connector snippet below is the shape of an automated reconciliation check.

In [ ]:
import snowflake.connector

conn = snowflake.connector.connect(
    account="acme-org_acme", user="MIGRATOR", role="MIGRATION_ROLE",
    warehouse="MIGRATION_WH",
)
cur = conn.cursor()

for source, target in [("migration.orders_raw", "analytics.fct_orders")]:
    cur.execute(f"SELECT COUNT(*) FROM {source}")
    before = cur.fetchone()[0]
    cur.execute(f"SELECT COUNT(*) FROM {target}")
    after = cur.fetchone()[0]
    print(source, target, before, after, "ok" if before == after else "MISMATCH")

```sql
-- Reconcile beyond row counts: sums, distinct keys, and null rates.
SELECT 'orders' AS entity,
       COUNT(*) AS rows_loaded,
       COUNT(DISTINCT order_id) AS distinct_keys,
       SUM(amount) AS total_amount,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customers
FROM analytics.fct_orders;

-- Track migration progress per subject area.
SELECT subject_area, status, source_rows, target_rows,
       CASE WHEN source_rows = target_rows THEN 'ok' ELSE 'investigate' END AS check_result
FROM migration.reconciliation_log
ORDER BY subject_area;
```

Migrate data before logic, let tooling translate syntax while you own the data model, permissions, and workload placement, and validate with sums, keys, and reports — not row counts alone.

Translate in two passes: first the mechanical syntax, which tooling such as SnowConvert handles, then the semantics, which no tool decides for you. Empty string versus null, integer division and rounding, timestamp precision and time zone handling, identifier case folding, and sequence or `IDENTITY` behavior all differ across engines, and each difference needs a test with real data rather than a code review.

Physical design does not translate directly. Redshift distribution and sort keys become load ordering plus an optional clustering key; Hive partition columns become directory layout and, for external or Iceberg tables, `PARTITION BY`; secondary indexes and `ANALYZE` have no equivalent because micro-partition statistics are automatic. Warehouse sizing is a separate decision from the original cluster shape.

Unquoted identifiers are folded to uppercase in Snowflake, so code that assumed lowercase names can break even when the SQL is otherwise valid. Reconcile with sums, distinct keys, and report-level comparisons rather than row counts alone, and keep the source running until a full reporting cycle passes.

## What are common Snowflake anti-patterns?

Most Snowflake performance and cost problems come from a short list of habits rather than from exotic failures. The first group is compute management: one always-on warehouse serving every workload, `AUTO_SUSPEND = 0`, auto-resume disabled so scheduled work fails silently, and reaching for a bigger warehouse or a clustering key before understanding the query. These are configuration mistakes, and they are usually the easiest to fix.

The second group is data layout and query shape. Loading files in arbitrary order so every filter becomes a full scan, over-clustering small or low-selectivity tables, selecting every column when only a few are needed, treating `VARIANT` as the schema instead of promoting queried fields into typed columns, and wrapping filter columns in functions. Each of these defeats an optimization that would otherwise be automatic.

The third group is ingestion and governance. Feeding thousands of tiny files into `COPY INTO` or a pipe, assuming Time Travel is a backup, and running with no resource monitors are all common. So is mixing DDL into long transactions, which cannot be rolled back as a unit because DDL auto-commits. None of these are inherent to Snowflake; they are patterns carried over from systems with different mechanics.

A practical habit is to check the profile and the account-usage views before changing anything. `PARTITIONS_SCANNED` versus `PARTITIONS_TOTAL` tells you whether it is a pruning problem, spill columns tell you whether it is a memory problem, queue time tells you whether it is a concurrency problem, and `WAREHOUSE_METERING_HISTORY` tells you what the change actually cost. The anti-pattern list below is worth re-reading whenever a query or a bill surprises you.

```text
- One always-on warehouse for every workload: separate loading, transformation, BI, and ad-hoc compute.
- AUTO_SUSPEND = 0 by default: you pay for idle compute.
- Auto-resume disabled: scheduled loads fail silently instead of waking the warehouse.
- Scaling up or clustering before fixing the query: an unordered load is a full scan at any size.
- Over-clustering: keys on small or low-selectivity tables cost maintenance and buy nothing.
- Thousands of tiny files into a pipe or COPY INTO: batch them first, or use Snowpipe Streaming.
- SELECT * everywhere: column pruning is one of the few levers that always works.
- Treating VARIANT as the schema: flatten queried fields into typed columns.
- Assuming Time Travel is backup: it is retention-bound and Fail-safe is not queryable.
- No resource monitors: cost grows invisibly until someone reads the bill.
- Long transactions mixing DDL and DML: DDL auto-commits independently and cannot be rolled back.
- Casting or applying functions to filter columns: it defeats micro-partition pruning.
- ORDER BY without LIMIT on a huge result: a sort that cannot be avoided and may spill.
```

```sql
-- A quick anti-pattern sweep over the last day of queries.
SELECT query_id, warehouse_name,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage,
       queued_overload_time / 1000 AS queued_s,
       total_elapsed_time / 1000 AS elapsed_s
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND (partitions_scanned > 10 * NULLIF(partitions_total / 100, 0)
       OR bytes_spilled_to_remote_storage > 0
       OR queued_overload_time > 5000)
ORDER BY total_elapsed_time DESC
LIMIT 50;
```

Most Snowflake pain comes from a short list of habits — idle compute, unordered loads, functions on filter columns, over-clustering, tiny files, and no monitors — and each one is fixed by configuration, ordering, or query shape rather than by a bigger warehouse.

Almost every item on the list defeats one of three mechanisms: micro-partition pruning, one of the caches, or workload isolation. A function on a filter column defeats pruning; an always-on shared warehouse defeats isolation and cache locality; tiny files defeat the load's parallelism. Framing a habit this way tells you which measurement will show the damage: the pruning ratio, the cache percentage, or the queue time.

Some patterns are anti-patterns only in the wrong context. `SELECT *` is fine inside a `CREATE TABLE AS SELECT` that genuinely needs every column, and a large warehouse is fine for a one-off backfill that must finish before a deadline. Judge by bytes read, credits per unit of work, and the latency actually required, not by style.

## How do you tune a slow Snowflake query, step by step?

Tuning in Snowflake is a diagnosis, not a guess. The workflow is always the same: find the expensive statement, get its `query_id`, read the numbers, open the query profile, identify the operator that dominated, and only then change something. Because every executed statement is recorded in `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY`, you can do the first half entirely in SQL and keep the evidence for a before-and-after comparison.

Start with the account-usage view. `TOTAL_ELAPSED_TIME` is wall-clock time from submission to completion and includes everything. `QUEUED_OVERLOAD_TIME` is time waiting for a free warehouse cluster, which points at concurrency rather than at the SQL. `QUEUED_PROVISIONING_TIME` is time spent waiting for compute to start, which points at auto-resume or resource-monitor suspension. `COMPILATION_TIME` is parsing and optimization, usually small but occasionally large for very complex queries. `EXECUTION_TIME` is the part a warehouse did actual work for, so it is the number to attack when queue and compile time are negligible.

The scan columns tell you whether the query read the right thing. `PARTITIONS_SCANNED` divided by `PARTITIONS_TOTAL` is the pruning ratio: near zero is excellent, above about 0.5 usually means the predicate could not prune. `PERCENTAGE_SCANNED_FROM_CACHE` tells you how much of the scan was served from cached data; a low value on a repeated query suggests the result cache missed and the warehouse cache is cold. The two spill columns are the memory signal: `BYTES_SPILLED_TO_LOCAL_STORAGE` means the warehouse ran out of memory for a sort, join, or aggregation and used disk, while `BYTES_SPILLED_TO_REMOTE_STORAGE` means it ran out of local disk too and that part of the query was much slower. `CREDITS_USED_CLOUD_SERVICES` shows any cloud-services cost the query incurred, which is normally small but flags metadata-heavy work.

Only after those numbers point somewhere do you open the profile. Map each finding to one fix: a high pruning ratio means filter or cluster; spill means a bigger warehouse or a leaner query; queue time means multi-cluster or workload separation; a huge row count from one join means fix the join; a long compile means simplify or precompute. Re-run the same statement and compare the same columns, because that is the only proof the change helped. The detailed recipes are in the sections that follow.

```sql
-- One query over QUERY_HISTORY with every diagnostic column that matters.
SELECT query_id,
       query_type,
       warehouse_name,
       warehouse_size,
       user_name,
       total_elapsed_time / 1000            AS elapsed_s,
       queued_overload_time / 1000          AS queue_overload_s,
       queued_provisioning_time / 1000      AS queue_provisioning_s,
       compilation_time / 1000              AS compile_s,
       execution_time / 1000                AS execution_s,
       partitions_scanned,
       partitions_total,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0), 3)      AS pruning_ratio,
       percentage_scanned_from_cache,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage,
       ROUND(bytes_spilled_to_local_storage  / POWER(1024, 3), 2) AS gb_spill_local,
       ROUND(bytes_spilled_to_remote_storage / POWER(1024, 3), 2) AS gb_spill_remote,
       credits_used_cloud_services
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND query_type IN ('SELECT', 'INSERT', 'MERGE')
  AND execution_time > 0
ORDER BY execution_time DESC
LIMIT 50;

-- Drill into one query once you have picked the query_id.
SELECT query_id, query_text,
       total_elapsed_time / 1000 AS elapsed_s,
       execution_time / 1000 AS execution_s,
       partitions_scanned, partitions_total,
       bytes_scanned, bytes_spilled_to_local_storage, bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE query_id = '<query_id>';

-- A quick triage view: classify the dominant symptom per query.
SELECT query_id, warehouse_name,
       total_elapsed_time / 1000 AS elapsed_s,
       CASE
         WHEN queued_overload_time > execution_time THEN 'concurrency: queueing'
         WHEN bytes_spilled_to_remote_storage > 0   THEN 'memory: remote spill'
         WHEN bytes_spilled_to_local_storage > 0    THEN 'memory: local spill'
         WHEN partitions_scanned > partitions_total * 0.5 THEN 'io: poor pruning'
         ELSE 'cpu: operator cost'
       END AS likely_cause
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND execution_time > 60000
ORDER BY execution_time DESC
LIMIT 50;
```

Tuning is `QUERY_HISTORY` to find and classify, the query profile to confirm the operator, one change, then re-run and compare the same columns.

There are two sources with different freshness. `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` holds a year of history but can lag by up to about 45 minutes. The `QUERY_HISTORY` table function in `INFORMATION_SCHEMA` is much closer to real time and is the right source while you are still iterating on a query. Use the account view for trends and attribution, and the table function for the statement you just ran.

Filter before you aggregate. A sweep over every query in a day reads a lot of rows and buries the signal; start from the workload's warehouse and a narrow time window, exclude internal statements, and only then sort by `EXECUTION_TIME` or by spill. Operator-level attribution exists only in the profile of the single `query_id`.

## How do you improve pruning and clustering for performance?

Pruning is the first performance lever because it reduces bytes read before any compute happens. The cheapest way to improve it is ordering at load time: if the data arrives sorted by the column your queries filter on most, each micro-partition covers a narrow range and a range predicate skips most of the table. A plain `INSERT ... SELECT ... ORDER BY` costs a sort once and pays for itself on every subsequent query, and it needs no clustering key at all.

When the table changes continuously and the natural order decays, a clustering key restores it. Design the key deliberately: start with the column used in range or equality filters that benefit from pruning, often a date, then append the highest-cardinality column that is also filtered, such as a customer or device id. Keys that are all low-cardinality, or that lead with a column nobody filters, buy little. Snowflake supports clustering keys with expressions, but keep the key short, because every extra expression multiplies the maintenance work.

Measure before and after with the system functions. `SYSTEM$CLUSTERING_INFORMATION` returns a JSON document describing the table's clustering, including `cluster_by`, `total_partition_count`, `total_constant_partition_count`, `average_overlaps`, `average_depth`, and a `partition_depth_histogram`. `average_depth` is the average number of partitions that overlap a point in the clustering space; lower is better and near 1 means near-perfect ordering. `average_overlaps` counts overlapping partitions, so a falling overlap count is the signal that reclustering is working. `SYSTEM$CLUSTERING_DEPTH` returns just the average depth for a table and optional key, which is handy for monitoring in a task.

Clustering is not free and not always appropriate. Automatic clustering runs as a serverless background process after DML, and `AUTOMATIC_CLUSTERING_HISTORY` records the credits it consumed. Do not cluster small tables, columns nobody filters, or tables whose queries are point lookups on a unique key, where search optimization is the better tool. Suspend reclustering while a table is being bulk loaded and resume it afterwards so maintenance does not run against a moving target, and drop the key entirely if the workload moves on.

```sql
-- Order on load: the cheapest pruning improvement there is.
INSERT INTO analytics.events (event_id, event_date, customer_id, payload)
SELECT event_id, event_date, customer_id, payload
FROM staging.events_raw
ORDER BY event_date, customer_id;

-- Design the key: filter date first, then the high-cardinality id.
ALTER TABLE analytics.events CLUSTER BY (event_date, customer_id);

-- Before and after, read the JSON that SYSTEM$CLUSTERING_INFORMATION returns.
SELECT SYSTEM$CLUSTERING_INFORMATION('analytics.events', '(event_date, customer_id)');

-- Pull individual fields out of that JSON for monitoring.
SELECT v:cluster_by::STRING                     AS cluster_by,
       v:total_partition_count::NUMBER          AS total_partitions,
       v:total_constant_partition_count::NUMBER AS constant_partitions,
       v:average_overlaps::FLOAT                AS average_overlaps,
       v:average_depth::FLOAT                   AS average_depth,
       v:partition_depth_histogram              AS depth_histogram
FROM (SELECT PARSE_JSON(
        SYSTEM$CLUSTERING_INFORMATION('analytics.events', '(event_date, customer_id)')
      ) AS v);

-- Just the depth number, cheap enough to run on a schedule.
SELECT SYSTEM$CLUSTERING_DEPTH('analytics.events', '(event_date, customer_id)') AS average_depth;

-- Pause maintenance during a bulk load, resume afterwards.
ALTER TABLE analytics.events SUSPEND RECLUSTERING;
-- ... load ...
ALTER TABLE analytics.events RESUME RECLUSTERING;

-- What automatic clustering cost, per table and day.
SELECT table_name, DATE_TRUNC('day', start_time) AS day,
       SUM(credits_used) AS credits
FROM snowflake.account_usage.automatic_clustering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY day DESC, credits DESC;
```

Prune first by ordering rows at load, add a clustering key of `(date, high_cardinality_id)` only for large repeatedly filtered tables, and judge it with `average_depth` and `average_overlaps` against the credits in `AUTOMATIC_CLUSTERING_HISTORY`.

Clustering and micro-partition statistics interact. Pruning uses the min/max ranges recorded per column per partition, and clustering changes those ranges by rewriting overlapping partitions. A clustering key is therefore worth it only when the rewritten ranges align with the predicates: a key whose leading column is never filtered leaves the ranges overlapping, keeps depth high, and buys nothing.

Depth is the metric to monitor, but the target is workload-specific. A table filtered on a narrow date range wants the date ranges of its partitions to be disjoint, so depth near 1 for that key. A table that is always scanned in full has no reason to be clustered at all, and improving its depth will not change the queries that matter.

Adding or changing a clustering key starts a background reorganization that can consume serverless credits for hours on a large table. Do it during a quiet window, suspend reclustering during a bulk load, and compare `AUTOMATIC_CLUSTERING_HISTORY` credits with the query improvement before keeping the key.

## How do you size and scale a warehouse for throughput?

Sizing is scale-up: a bigger warehouse has more nodes and more parallel workers, so a single parallelizable query finishes sooner at roughly the same total credits. Scaling out is a multi-cluster warehouse: several identical clusters behind one name, so many concurrent queries run at once without waiting. Pick scale-up when one query is latency-bound, and scale-out when lots of queries are throughput-bound. The two solve different problems and cost differently.

Concurrency behavior in a multi-cluster warehouse is governed by `SCALING_POLICY` and `MAX_CONCURRENCY_LEVEL`. `MAX_CONCURRENCY_LEVEL` is the number of queries a cluster is expected to handle before Snowflake starts another cluster; lower values make the warehouse add clusters sooner. `STANDARD` scales up quickly when queries queue and scales down when load drops, which suits interactive BI where latency matters. `ECONOMY` scales up slowly and keeps clusters for several idle minutes, which suits batch jobs that would rather wait than pay for cluster churn. Both policies respect `MIN_CLUSTER_COUNT` and `MAX_CLUSTER_COUNT`.

Use the load-history table function and `QUERY_HISTORY` to decide. `INFORMATION_SCHEMA.WAREHOUSE_LOAD_HISTORY` reports `AVG_RUNNING`, `AVG_QUEUED_LOAD`, `AVG_QUEUED_PROVISIONING`, and `AVG_BLOCKED` per warehouse over time; sustained `AVG_QUEUED_LOAD` means queries wait for compute, which is the case for more clusters or a bigger cluster. `QUEUED_OVERLOAD_TIME` in `QUERY_HISTORY` isolates the same signal per query. If execution time is long but queue time is zero, adding clusters will not help — the queries themselves need to get cheaper.

Choose sizes by workload. Dashboards and development: XSMALL to SMALL. Standard ELT: MEDIUM to LARGE. Heavy transformations and backfills: LARGE to XLARGE. Beyond that, suspect the query. On the cost side, a smaller warehouse is slower per query but costs the same per credit-hour, so per unit of work a well-tuned small warehouse is often cheapest; the price of small is latency and concurrency. Auto-suspend of 60 seconds suits batch, 300 to 600 seconds suits interactive use, and `AUTO_RESUME` should stay on so scheduled work can wake the warehouse instead of failing.

```sql
-- Scale up: more nodes for one latency-bound query.
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';

-- Scale out: more clusters for concurrency, with a policy and a concurrency target.
ALTER WAREHOUSE bi_wh SET
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 8
  SCALING_POLICY = 'STANDARD'
  MAX_CONCURRENCY_LEVEL = 4;

ALTER WAREHOUSE batch_wh SET
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 4
  SCALING_POLICY = 'ECONOMY'
  MAX_CONCURRENCY_LEVEL = 8;

SHOW WAREHOUSES;

-- Queueing versus running load: is throughput the problem?
SELECT start_time, warehouse_name, avg_running, avg_queued_load,
       avg_queued_provisioning, avg_blocked
FROM TABLE(information_schema.warehouse_load_history(
  warehouse_name => 'BI_WH',
  date_range_start => DATEADD('hour', -12, CURRENT_TIMESTAMP())))
ORDER BY start_time;

-- Per-query evidence: long queue time means scale out, long execution means fix the query.
SELECT warehouse_name,
       COUNT(*) AS queries,
       ROUND(SUM(queued_overload_time) / 1000, 1) AS queued_s,
       ROUND(SUM(execution_time) / 1000, 1)       AS execution_s
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY queued_s DESC;
```

Scale up for latency and out for concurrency, drive the choice from `AVG_QUEUED_LOAD` and `QUEUED_OVERLOAD_TIME`, and remember that a smaller warehouse is slower per query but no more expensive per credit.

`MAX_CONCURRENCY_LEVEL` sets how many queries a cluster is expected to handle before Snowflake starts another. Lowering it makes the warehouse scale out earlier and stay wider; raising it packs more work per cluster and delays extra clusters. It is a dial between queueing and credit burn, not a correctness setting.

Make sizing decisions from the distribution of query runtimes, not the average. If p50 is fast and p95 is slow, the warehouse is probably sized for the median and the outliers are queueing behind it; if most queries are slow, the problem is the SQL or the table, and neither scale-up nor scale-out fixes it.

A warehouse keeps the size it had when a query started, so a resize during a batch affects only later statements. Scale between batches and measure the next batch, not the one in flight.

## How do you diagnose and fix spilling?

Spilling happens when an operation needs more memory than the warehouse has. Snowflake first writes the overflow to local disk, recorded as `BYTES_SPILLED_TO_LOCAL_STORAGE`. If the local disk fills too, it writes to remote storage, recorded as `BYTES_SPILLED_TO_REMOTE_STORAGE`. Local spill is a slowdown; remote spill is a severe slowdown, because every spilled row now travels over the network. The three usual culprits are a large sort, a large hash join, and a large aggregation, all of which build in-memory structures proportional to the data they process.

You can see spilling in two places. In the query profile, the offending operator — typically `Sort`, `Join`, or `Aggregate` — shows bytes spilled in its statistics, and the profile's overall section shows the query total. In SQL, `QUERY_HISTORY` records the same two columns per query, so a single query can find the worst offenders. Remote spill is the clearer signal: if it is non-zero, the warehouse could not even hold the intermediate state on local disk.

The fixes are ordered by how much they change. The quickest is a bigger warehouse, because a larger size has more memory per node and may eliminate the spill entirely, at roughly the same credits per unit of work. The better fix is usually to reduce what must be held in memory: filter earlier and more aggressively, pre-aggregate before the join, remove an unnecessary `ORDER BY` or `DISTINCT`, avoid `SELECT *` so less data flows through every operator, and check for exploding joins where a key is not unique on both sides and row counts multiply. If those do not help, split the query into stages and materialize an intermediate result so no single statement has to hold everything at once.

Read remote spill as a statement about the whole query, not just one operator. It often means several operators each spilled, or that a join produced far more rows than either input, or that the query is simply shaped wrong for a warehouse of any reasonable size. Treating it as 'buy a bigger warehouse' works until the data grows again, which is why the structural fixes come first on any recurring pipeline.

```sql
-- Find the biggest spillers in the last day.
SELECT query_id, warehouse_name, warehouse_size,
       total_elapsed_time / 1000 AS elapsed_s,
       ROUND(bytes_spilled_to_local_storage  / POWER(1024, 3), 2) AS gb_local,
       ROUND(bytes_spilled_to_remote_storage / POWER(1024, 3), 2) AS gb_remote
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND bytes_spilled_to_local_storage + bytes_spilled_to_remote_storage > 0
ORDER BY bytes_spilled_to_remote_storage DESC, bytes_spilled_to_local_storage DESC
LIMIT 25;

-- Aggregate spill by warehouse to see whether it is systematic.
SELECT warehouse_name,
       COUNT(*) AS spilling_queries,
       ROUND(SUM(bytes_spilled_to_local_storage)  / POWER(1024, 4), 2) AS tb_local,
       ROUND(SUM(bytes_spilled_to_remote_storage) / POWER(1024, 4), 2) AS tb_remote
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
HAVING spilling_queries > 0
ORDER BY tb_remote DESC, tb_local DESC;
```

```sql
-- Before: a join on a non-unique key plus a late filter and an unnecessary sort.
SELECT o.customer_id, SUM(o.amount) AS revenue
FROM analytics.orders o
JOIN analytics.events e ON e.customer_id = o.customer_id
WHERE o.order_date >= '2026-01-01'
GROUP BY o.customer_id
ORDER BY revenue DESC;

-- After: filter and aggregate before the join, drop the sort.
WITH recent_orders AS (
  SELECT customer_id, SUM(amount) AS revenue
  FROM analytics.orders
  WHERE order_date >= '2026-01-01'
  GROUP BY customer_id
),
distinct_customers AS (
  SELECT DISTINCT customer_id FROM analytics.events
)
SELECT r.customer_id, r.revenue
FROM recent_orders r
JOIN distinct_customers c ON c.customer_id = r.customer_id;

-- If spill persists, give the operation more memory.
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';
```

Spilling means an operation outgrew warehouse memory — local spill is a slowdown, remote spill a severe one — so reduce what is held in memory first and raise the warehouse size second.

Spilling is reported per operator and summed per query. Local spill writes to the node's SSD; remote spill goes to object storage over the network and is an order of magnitude slower. Because a query can have several spilling operators, the query-level total can hide which one dominates; the profile shows it per node.

Any operator that builds a hash table or sorts a full input can spill: `JOIN`, `GROUP BY`, `ORDER BY`, `DISTINCT`, and window functions. Reducing rows before those operators is more effective than adding memory, because it shrinks the structure that must fit and also reduces the work every downstream operator does.

A bigger warehouse can hide a structural problem until the data grows. If a query spills again after one size increase, stop resizing and reshape the query, or the next data volume will spill on the larger warehouse too.

## Which query patterns make Snowflake slow?

Snowflake's optimizer does a lot automatically, but a handful of SQL patterns defeat it. The most common is applying a function or cast to the column in a filter, such as `WHERE CAST(order_date AS STRING) = '2026-10-04'` or `WHERE YEAR(order_date) = 2026`. The predicate no longer references the raw column, so micro-partition min/max statistics cannot be used and the query scans the table. Rewrite the filter to compare the raw column to a literal range.

The second cluster of problems is about how much data moves. `SELECT *` pulls every column through every operator, which hurts scans, joins, and especially sorts. `DISTINCT` and `UNION` force deduplication and sorting where `UNION ALL` would simply concatenate. `ORDER BY` without `LIMIT` makes the warehouse sort the entire result, often spilling, when the caller only wanted the top rows. Wide `VARIANT` extraction repeats the same path expressions across millions of rows instead of promoting the field once into a typed column.

The third group is join and concurrency shape. Cross joins and joins on non-unique keys multiply rows and turn a small aggregation into a huge one. Implicit conversion between `VARCHAR` and numeric columns in a join prevents the optimizer from using statistics and can force a broadcast or a full scan. Many small concurrent queries, such as a dashboard issuing dozens of single-row lookups, queue against each other on one warehouse even though each query is trivial.

Finally, repeated work that could be cached or precomputed is wasted work. A report that re-scans a large table every time the dashboard refreshes ignores the result cache, and a repeated expensive aggregation that never changes between loads could be a materialized view. Before rewriting SQL, confirm which pattern applies with `EXPLAIN` and the profile, then apply the smallest change and measure. The before-and-after pairs below are the ones worth memorizing.

```sql
-- 1. Function on the filter column: loses pruning.
SELECT COUNT(*) FROM analytics.orders WHERE YEAR(order_date) = 2026;
-- Rewrite as a raw range comparison.
SELECT COUNT(*) FROM analytics.orders
WHERE order_date >= '2026-01-01' AND order_date < '2027-01-01';

-- 2. Column list instead of SELECT *: less data through every operator.
SELECT * FROM analytics.orders WHERE order_date = '2026-10-04';
SELECT order_id, customer_id, amount FROM analytics.orders WHERE order_date = '2026-10-04';

-- 3. UNION deduplicates and sorts; UNION ALL does not.
SELECT customer_id FROM analytics.orders_2025
UNION
SELECT customer_id FROM analytics.orders_2026;
SELECT customer_id FROM analytics.orders_2025
UNION ALL
SELECT customer_id FROM analytics.orders_2026;

-- 4. ORDER BY without LIMIT sorts the whole result; add a bound when you only need the top rows.
SELECT customer_id, SUM(amount) AS revenue FROM analytics.orders
GROUP BY customer_id ORDER BY revenue DESC;
SELECT customer_id, SUM(amount) AS revenue FROM analytics.orders
GROUP BY customer_id ORDER BY revenue DESC LIMIT 100;

-- 5. Implicit VARCHAR-to-number join prevents a clean join plan.
SELECT * FROM analytics.orders o JOIN analytics.customers c ON o.customer_id = c.customer_id_str;
-- Cast explicitly once, before the join.
SELECT * FROM analytics.orders o
JOIN (SELECT TO_NUMBER(customer_id_str) AS customer_id FROM analytics.customers) c
  ON o.customer_id = c.customer_id;

-- 6. Distinct where the grain is already unique.
SELECT DISTINCT order_id, customer_id FROM analytics.orders;
SELECT order_id, customer_id FROM analytics.orders;

-- 7. Repeated VARIANT extraction: promote the field once into a typed view.
SELECT payload:user.id::NUMBER, COUNT(*) FROM raw.events GROUP BY 1;
CREATE OR REPLACE VIEW analytics.v_events AS
  SELECT payload:user.id::NUMBER AS user_id FROM raw.events;
SELECT user_id, COUNT(*) FROM analytics.v_events GROUP BY 1;
```

Avoid functions on filter columns, `SELECT *`, needless `DISTINCT` and `ORDER BY`, implicit type conversions, exploding joins, and repeated wide extraction — every one of them defeats an optimization Snowflake would otherwise apply for free.

The optimizer pushes filters to the scan and chooses join order from statistics. Anything that hides a literal or a type from it removes information it needs: a function, a cast, an implicit conversion, a `LIKE` with a leading wildcard, or a `DISTINCT` where the grain is already unique. Rewriting for the optimizer usually means making the predicate more literal and the plan more obvious, not making the query shorter.

Repeated work is a pattern too. Recomputing a stable aggregate on every dashboard refresh, re-scanning a table that changed in one partition, or extracting the same JSON path in every query spends credits on results that already exist. Before rewriting, ask whether the answer could be cached, materialized, or read from a clustered or optimized access path.

## How do you make repeated aggregations fast?

When the same aggregation runs again and again, the answer is almost never a bigger warehouse. Snowflake offers four mechanisms, and choosing among them is a matter of matching the tool to the shape of the repetition. The first is the result cache: an identical query that touches unchanged tables returns the stored result for up to 24 hours at no compute cost. It is the cheapest option when query text is truly identical and the underlying tables are not being written.

The second is a materialized view, which stores a precomputed aggregate and is maintained automatically as the base table changes. It has strict limits: a single base table, no joins, no `HAVING`, no window functions, and a limited set of aggregates. Within those limits the optimizer can transparently rewrite an eligible query to read it. Use it for a few expensive aggregations on a moderately changing table, and watch maintenance credits.

The third is the search optimization service, which is about selective lookups rather than aggregations. It makes equality and `IN` predicates, leading-wildcard `LIKE`, `ARRAY_CONTAINS`, and geospatial predicates fast where pruning cannot help, and it is maintained on every DML. Add it to a large, slowly changing table where users run point lookups by email, id, or a substring.

The fourth is the query acceleration service, which offloads part of a large scan and aggregation to serverless compute for eligible queries. It suits a warehouse sized for the median query that occasionally receives a much larger one. Choose by asking what is repeated: the exact same query (result cache), the same single-table aggregate over changing data (materialized view), selective point lookups (search optimization), or an occasional outlier scan (query acceleration). Each is metered differently, so measure the credits against the time saved.

```sql
-- 1. Result cache: identical text over unchanged tables is served for free.
SELECT order_date, SUM(amount) FROM analytics.orders GROUP BY order_date;
SELECT order_date, SUM(amount) FROM analytics.orders GROUP BY order_date;  -- cached

-- 2. Materialized view: single table, no joins, maintained automatically.
CREATE OR REPLACE MATERIALIZED VIEW analytics.mv_daily_revenue AS
SELECT order_date, SUM(amount) AS revenue, COUNT(*) AS orders
FROM analytics.orders
GROUP BY order_date;

-- 3. Search optimization: selective lookups, not aggregations.
ALTER TABLE analytics.customers ADD SEARCH OPTIMIZATION ON EQUALITY(email);
SELECT customer_id FROM analytics.customers WHERE email = 'someone@example.com';

-- 4. Query acceleration: absorb an occasional much larger scan.
ALTER WAREHOUSE bi_wh SET ENABLE_QUERY_ACCELERATION = TRUE
  QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;
SELECT SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>');

-- Look before you choose: how often is the exact query text repeated?
SELECT MD5(query_text) AS query_fingerprint,
       COUNT(*) AS runs,
       ROUND(AVG(total_elapsed_time) / 1000, 1) AS avg_elapsed_s,
       ROUND(SUM(total_elapsed_time) / 1000, 1) AS total_elapsed_s
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY query_fingerprint
HAVING runs > 10
ORDER BY total_elapsed_s DESC
LIMIT 20;
```

Match the mechanism to the repetition: result cache for identical queries, materialized views for single-table aggregates, search optimization for selective lookups, and query acceleration for occasional outlier scans.

The four mechanisms differ in what invalidates them. The result cache is invalidated by any DML on the source tables and by a changed session parameter that is part of the cache key, and it needs identical text. A materialized view survives DML because it is maintained, but it covers a single table. Search optimization is maintained incrementally by DML and only helps selective predicates. Query acceleration caches nothing; it adds compute to the scan.

Start from the cheapest. If the exact query repeats over unchanged tables, the result cache costs nothing and needs no object. If the repetition is the same single-table aggregate over changing data, a materialized view. If it is a point lookup, search optimization. Only an occasional much larger scan justifies acceleration. Measure credits per refresh before and after in every case.

## How do you load data fast?

Load speed in Snowflake is mostly a file-shape problem. `COPY INTO` and Snowpipe parallelize across files, so the number and size of files determine how much of the warehouse or service can be used at once. Snowflake recommends compressed files of roughly 100 to 250 MB. Too many tiny files waste parallelism and flood the load metadata; a single huge file cannot be split effectively and becomes close to a single-threaded load. Compacting files in the landing zone before loading is one of the highest-leverage ingestion fixes.

Batch your loads deliberately. `PATTERN` limits a load to the files for a given date or prefix so you can load incrementally and debug a small slice. `ON_ERROR` decides how bad rows are handled: `ABORT_STATEMENT` fails the load, `CONTINUE` loads good rows and records errors, and `SKIP_FILE` skips a bad file. `VALIDATION_MODE` performs a dry run and returns errors without loading, which is the safe way to test a new format. `PURGE = TRUE` removes staged files after success, and `FORCE = FALSE` (the default) uses Snowflake's 64-day load metadata to skip files already loaded.

Snowpipe versus batch is a latency-versus-cost decision. Snowpipe ingests files continuously as they arrive using serverless compute and charges per file, which suits steady, low-latency arrival. A scheduled `COPY INTO` on a warehouse is usually cheaper per byte when files arrive in large predictable waves, because the warehouse loads many files in parallel and then suspends. Snowpipe Streaming goes further and writes rows directly, removing files from the path entirely for row-level latency.

The gotchas are duplicate handling and visibility. Snowpipe and Snowpipe Streaming are at-least-once, so dedupe on a business key or file name. Avoid single-threaded loads such as loading one file at a time from a script, one row per `INSERT`, or unloading to a single output file. Finally, watch `COPY_HISTORY` for `error_count` and `status`, and `PIPE_USAGE_HISTORY` for serverless credits, because a silently failing pipe looks like data that simply never arrived.

```sql
-- Compact and load a bounded slice, tolerating bad rows, cleaning up after success.
COPY INTO raw.orders.orders
FROM @raw.orders.landing
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
PATTERN = '.*2026-10-04.*[.]parquet'
ON_ERROR = 'CONTINUE'
PURGE = TRUE
FORCE = FALSE;

-- Dry run before committing to a new format or feed.
COPY INTO raw.orders.orders FROM @raw.orders.landing
  FILE_FORMAT = (TYPE = PARQUET)
  VALIDATION_MODE = 'RETURN_ERRORS';

-- Unload into many appropriately sized files instead of one giant file.
COPY INTO @raw.orders.exports/daily/
FROM (SELECT * FROM analytics.marts.daily_revenue)
FILE_FORMAT = (TYPE = PARQUET)
MAX_FILE_SIZE = 100000000;

-- Load throughput and failures, by table and day.
SELECT DATE_TRUNC('day', last_load_time) AS load_day,
       table_name,
       COUNT(*) AS files,
       SUM(row_count) AS rows_loaded,
       SUM(error_count) AS errors
FROM snowflake.account_usage.copy_history
WHERE last_load_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY load_day DESC, rows_loaded DESC;

-- Any file that failed and why.
SELECT file_name, status, first_error_message, last_load_time
FROM snowflake.account_usage.copy_history
WHERE status = 'LOAD_FAILED'
  AND last_load_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY last_load_time DESC;
```

Load speed is file shape: aim for 100 to 250 MB compressed files, batch with `PATTERN`, use `VALIDATION_MODE` and a deliberate `ON_ERROR`, choose Snowpipe for latency and batch for cost, and watch `COPY_HISTORY`.

Load throughput is bounded by the slowest of three things: how many files can be processed in parallel, how fast each file can be read, and how fast the warehouse can write the resulting partitions. Very small files hurt the first and second, and a single huge file hurts the second, which is why `MAX_FILE_SIZE` on unload and file compaction before load attack the same bottleneck.

The two ingestion paths bill differently. `COPY INTO` runs on a warehouse you manage, so you control its size and lifecycle, and loading in large scheduled batches lets it suspend between runs. Snowpipe and Snowpipe Streaming run on Snowflake-managed serverless compute billed as credits attributed to the pipe or the streaming client, which is convenient for continuous arrival but less predictable per byte.

## How do you tune without burning credits?

Tuning has a cost, so treat it as an experiment with a budget. The measurement layer is `SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY`, which records credits per warehouse per hour split into compute and cloud services, and `QUERY_ATTRIBUTION_HISTORY`, which attributes credits to individual queries and to the tags on them. Set a `QUERY_TAG` before a tuning run so the before and after queries are easy to isolate, and compare the same query text so the difference is the change you made rather than the data.

Guardrails come first. Attach a resource monitor to the account and to any warehouse you are experimenting with, with `DO NOTIFY` early and `DO SUSPEND` at the quota. Set `AUTO_SUSPEND` to a short interval on test warehouses so an abandoned experiment does not run all night, and set `INITIALLY_SUSPENDED = TRUE` when creating a warehouse so it does not start until you use it. Right-sizing experiments should always start from a measured baseline and move one step at a time, because the useful comparison is credits per query, not credits per hour.

Define the metric before you start. For a pipeline, that is credits per load and total wall-clock time. For a dashboard, it is credits per refresh and p95 latency. Then change one thing — ordering, a clustering key, a warehouse size, a rewritten join — and re-run the same workload. A change that halves latency but doubles credits is not automatically good; a change that halves credits at the same latency usually is. Where the improvement is structural, such as fixing pruning, it usually wins on both axes.

A short checklist covers most tuning sessions. Confirm the query is not simply queued. Confirm pruning with the scanned-to-total ratio. Check spill. Remove functions from filter columns and `SELECT *` from hot queries. Confirm the warehouse actually suspends between runs. Attribute credits to a query tag so the result is provable. If nothing improves, stop and question whether the workload belongs on a warehouse at all, because the cheapest query is the one you precompute or cache.

```sql
-- Baseline and after, attributed to a query tag.
ALTER SESSION SET QUERY_TAG = 'tune:pruning_2026_10';

SELECT warehouse_name,
       DATE_TRUNC('hour', start_time) AS hour,
       SUM(credits_used_compute) AS compute_credits,
       SUM(credits_used_cloud_services) AS cloud_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY hour DESC;

-- Credits per query and per tag: the number that decides whether tuning paid.
SELECT query_tag,
       COUNT(*) AS queries,
       SUM(credits_used) AS credits,
       ROUND(SUM(credits_used) / NULLIF(COUNT(*), 0), 6) AS credits_per_query
FROM snowflake.account_usage.query_attribution_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY query_tag
ORDER BY credits DESC;

-- Guardrails while experimenting.
CREATE OR REPLACE RESOURCE MONITOR tune_monitor WITH CREDIT_QUOTA = 50
  FREQUENCY = DAILY
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
ALTER WAREHOUSE tune_wh SET RESOURCE_MONITOR = tune_monitor
  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
```

```text
Tuning checklist
1. Is the query queued rather than slow? Check QUEUED_OVERLOAD_TIME.
2. Is it pruning? Compare PARTITIONS_SCANNED with PARTITIONS_TOTAL.
3. Is it spilled? Local spill is slow, remote spill is severe.
4. Are functions or casts on the filter column?
5. Is SELECT * pulling columns nothing uses?
6. Is the warehouse actually suspending between runs?
7. Did credits per query improve, not just latency?
8. Can the result be cached or precomputed instead of re-run?
```

Measure with `WAREHOUSE_METERING_HISTORY` and `QUERY_ATTRIBUTION_HISTORY`, tag the run, guard it with a resource monitor, and judge every change by credits per query — not by latency alone.

Attribution is what makes a tuning session provable. Setting a `QUERY_TAG` before the run lets `QUERY_ATTRIBUTION_HISTORY` group the before and the after, and `WAREHOUSE_METERING_HISTORY` shows the warehouse-level effect. Without a tag, the comparison is a guess against everything else that ran in the same hour.

Do the free checks first: was the query queued, did it prune, did it spill, did the warehouse actually suspend between runs. Only after those come changes that cost something to test, such as a clustering key or a resize, and those should run against a resource monitor with a low daily quota.

## Tuning exercises

These exercises use only the Python standard library and model the tuning ideas with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The exercises model the tuning arithmetic in the same units the platform uses. Credits are `credits-per-hour x seconds / 3600`, which is why the scale-up exercise comes out credit-neutral: doubling the size doubles the rate and roughly halves the time. The partition-ratio and spill classifiers mirror the columns you would read from `QUERY_HISTORY`, and the multi-cluster simulation models the tradeoff between queue time and cluster uptime that `STANDARD` and `ECONOMY` make in practice.

Treat the asserted numbers as a check on the formula rather than a prediction of a real account. Real runs add variance from caching, concurrency, and data skew, so the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Is this a pruning problem?

**Task.** Given the metrics for one query, compute the scanned-partition ratio and decide whether the query has a pruning problem. A query is considered adequately pruned when it scans at most half of the partitions.

**Hint.** `ratio = partitions_scanned / partitions_total`; a ratio above 0.5 means pruning is poor.

**Expected result.** `0.960 (96.0%)` scanned and the verdict `fix pruning`.

In [ ]:
metrics = {"partitions_scanned": 4800, "partitions_total": 5000}

ratio = metrics["partitions_scanned"] / metrics["partitions_total"]
verdict = "fix pruning" if ratio > 0.5 else "pruning is acceptable"

print(f"scanned ratio = {ratio:.3f} ({ratio * 100:.1f}%)")
print("verdict:", verdict)

assert round(ratio, 3) == 0.96
assert verdict == "fix pruning"

### Try it 2. Scale up or scale out?

**Task.** Using a credits-per-hour table, compute the credits consumed by a 600-second query on a MEDIUM warehouse and by the same query taking 300 seconds on a LARGE warehouse. Then compute the cost of running three SMALL clusters for a full hour and compare it with one MEDIUM cluster for a full hour.

**Hint.** `credits = credits_per_hour * seconds / 3600`. Doubling the size should roughly halve the runtime at about the same credits.

**Expected result.** The MEDIUM/600s and LARGE/300s runs each cost about `0.667` credits, so scale-up is credit-neutral. Three SMALL clusters cost `6.0` credits per hour versus `4.0` for one MEDIUM.

In [ ]:
CREDITS_PER_HOUR = {"XSMALL": 1, "SMALL": 2, "MEDIUM": 4, "LARGE": 8, "XLARGE": 16}


def credits(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600


medium_run = credits("MEDIUM", 600)
large_run = credits("LARGE", 300)
print(f"MEDIUM 600s = {medium_run:.3f} credits")
print(f"LARGE  300s = {large_run:.3f} credits")
assert abs(medium_run - large_run) < 1e-9

scale_out = 3 * credits("SMALL", 3600)
scale_up = 1 * credits("MEDIUM", 3600)
print(f"3 x SMALL for 1h = {scale_out:.1f} credits")
print(f"1 x MEDIUM for 1h = {scale_up:.1f} credits")
assert scale_out == 6.0 and scale_up == 4.0

### Try it 3. Detect remote spill

**Task.** Write a function that classifies a metrics dictionary as `no spill`, `local spill`, or `remote spill`, then classify three example queries. Remote spill is the severe case because it means local disk was exhausted too.

**Hint.** Check `bytes_spilled_to_remote_storage` first; it takes precedence over local spill. Use `.get(key, 0)` so missing columns behave as zero.

**Expected result.** The three examples classify as `local spill`, `remote spill`, and `no spill`.

In [ ]:
def classify_spill(metrics):
    local = metrics.get("bytes_spilled_to_local_storage", 0)
    remote = metrics.get("bytes_spilled_to_remote_storage", 0)
    if remote > 0:
        return "remote spill"
    if local > 0:
        return "local spill"
    return "no spill"


local_only = {"bytes_spilled_to_local_storage": 2_000_000_000,
              "bytes_spilled_to_remote_storage": 0}
remote_too = {"bytes_spilled_to_local_storage": 1_000_000_000,
              "bytes_spilled_to_remote_storage": 500_000_000}
clean = {"bytes_spilled_to_local_storage": 0, "bytes_spilled_to_remote_storage": 0}

for name, m in [("query_a", local_only), ("query_b", remote_too), ("query_c", clean)]:
    print(name, "->", classify_spill(m))

assert classify_spill(local_only) == "local spill"
assert classify_spill(remote_too) == "remote spill"
assert classify_spill(clean) == "no spill"

### Try it 4. Choose a clustering key

**Task.** Given the approximate cardinality of four columns, pick a two-column clustering key that leads with the date used by range filters and then uses the highest-cardinality filtered column.

**Hint.** Keep the range column first, then take the column with the largest distinct count among the remaining candidates. `max(iterable, key=...)` selects it in one pass.

**Expected result.** The chosen key is `('order_date', 'customer_id')`.

In [ ]:
cardinality = {
    "order_date": 90,          # low cardinality, but range filters use it
    "customer_id": 2_000_000,  # high cardinality, also filtered
    "country": 40,             # nobody filters on it
    "event_type": 12,
}
range_column = "order_date"

id_column = max((c for c in cardinality if c != range_column), key=cardinality.get)
cluster_key = (range_column, id_column)
print("cluster key:", cluster_key)
print(f"{id_column} has {cardinality[id_column]:,} distinct values")

assert cluster_key == ("order_date", "customer_id")

### Try it 5. Multi-cluster scaling: STANDARD versus ECONOMY

**Task.** Simulate twelve ticks of arriving queries on a multi-cluster warehouse with at most three clusters. STANDARD adds a cluster as soon as queries queue; ECONOMY adds one only after three consecutive ticks with a non-empty queue. Compare total queue length with total cluster-ticks.

**Hint.** Each tick: finish work, add arrivals, apply the scaling policy, then start as many queries as there is spare capacity. Sum the queue length and the cluster count at the end of each tick.

**Expected result.** STANDARD keeps the queue at zero but pays 35 cluster-ticks; ECONOMY pays 29 cluster-ticks and leaves a queue of 14 query-ticks.

In [ ]:
def simulate(policy, ticks=12, arrivals_per_tick=1, runtime=3, max_clusters=3):
    clusters = 1
    active = []          # remaining ticks for each running query
    queue = 0
    queue_ticks = 0
    cluster_ticks = 0
    patient = 0          # consecutive ticks with a non-empty queue (ECONOMY only)
    for _ in range(ticks):
        active = [r - 1 for r in active if r - 1 > 0]
        queue += arrivals_per_tick
        if policy == "STANDARD":
            if queue > 0:
                clusters = min(max_clusters, clusters + 1)
        else:
            if queue > 0:
                patient += 1
                if patient % 3 == 0:
                    clusters = min(max_clusters, clusters + 1)
            else:
                patient = 0
        capacity = max(clusters - len(active), 0)
        started = min(capacity, queue)
        active += [runtime] * started
        queue -= started
        queue_ticks += queue
        cluster_ticks += clusters
    return queue_ticks, cluster_ticks


standard = simulate("STANDARD")
economy = simulate("ECONOMY")
print(f"STANDARD: queue_ticks={standard[0]:2d}  cluster_ticks={standard[1]}")
print(f"ECONOMY : queue_ticks={economy[0]:2d}  cluster_ticks={economy[1]}")

assert standard[0] < economy[0], "STANDARD should queue less"
assert standard[1] > economy[1], "STANDARD should pay for more clusters"

### Try it 6. Estimate cost per query

**Task.** Given a price of 3.00 currency units per credit, compute the credits and the cost of four queries and the average cost per query.

**Hint.** Cost is `credits * price`, and credits are `credits_per_hour * seconds / 3600`.

**Expected result.** Total cost is about `1.72`, an average of about `0.43` per query.

In [ ]:
PRICE_PER_CREDIT = 3.00
CREDITS_PER_HOUR = {"SMALL": 2, "MEDIUM": 4, "LARGE": 8, "XLARGE": 16}
queries = [("SMALL", 12), ("MEDIUM", 30), ("LARGE", 120), ("XLARGE", 60)]


def query_cost(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600 * PRICE_PER_CREDIT


costs = []
for size, seconds in queries:
    cost = query_cost(size, seconds)
    costs.append(cost)
    print(f"{size:<7} {seconds:>4}s -> {cost:.4f}")

total = sum(costs)
print(f"total = {total:.4f}, average = {total / len(costs):.4f}")
assert round(total, 2) == 1.72
assert round(total / len(costs), 2) == 0.43

### Try it 7. Read a cache-hit column

**Task.** Given the `percentage_scanned_from_cache` values for five runs of the same query, report the average and decide whether most runs were cache-served. Treat a run as cache-served when the value is above 90 percent.

**Hint.** Count the runs above the threshold and compare with half the number of runs.

**Expected result.** The average is `63.0` percent and most runs (3 of 5) were cache-served.

In [ ]:
scanned_from_cache = [98.5, 12.0, 99.1, 8.4, 97.0]

average = sum(scanned_from_cache) / len(scanned_from_cache)
cache_served = [value for value in scanned_from_cache if value > 90]
mostly_cached = len(cache_served) >= len(scanned_from_cache) / 2

print(f"average from cache = {average:.1f}%")
print(f"cache-served runs = {len(cache_served)} of {len(scanned_from_cache)}")
print("mostly cached:", mostly_cached)

assert round(average, 1) == 63.0
assert len(cache_served) == 3
assert mostly_cached is True

### Try it 8. Decide whether a scale-up experiment paid off

**Task.** A query took 60 seconds on a SMALL warehouse and 32 seconds on a MEDIUM warehouse. Compute the credits for both, the speedup, and decide whether the change is worth keeping. Treat it as worth keeping when the credits rise by no more than 10 percent and the speedup is at least 1.5x.

**Hint.** Compare credits, not size names, and remember that the smaller warehouse is not cheaper per credit-hour.

**Expected result.** Speedup is about `1.88x`, credits rise about `6.7` percent, so the scale-up is worth keeping.

In [ ]:
CREDITS_PER_HOUR = {"SMALL": 2, "MEDIUM": 4, "LARGE": 8}


def credits(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600


before = credits("SMALL", 60)
after = credits("MEDIUM", 32)
speedup = 60 / 32
credit_change = (after - before) / before
worth_keeping = credit_change <= 0.10 and speedup >= 1.5

print(f"SMALL 60s  = {before:.4f} credits")
print(f"MEDIUM 32s = {after:.4f} credits")
print(f"speedup = {speedup:.2f}x, credit change = {credit_change * 100:+.1f}%")
print("worth keeping:", worth_keeping)

assert round(speedup, 2) == 1.88
assert round(credit_change * 100, 1) == 6.7
assert worth_keeping is True

## Real world use cases for Snowflake

The sections below put the pieces together. Each one describes a realistic workload, the Snowflake features it leans on, and why those choices hold up in production. The common thread is that the architecture decisions — workload separation, ordering, governance, and cost guardrails — matter more than any single feature.

The same primitives appear in every scenario: separate compute per workload, a load order or clustering key that matches the dominant filter, change capture with streams or dynamic tables, and governance enforced in the engine. When a design starts to feel complicated, check whether it is re-implementing one of those four things outside Snowflake; if so, the native feature is usually cheaper and easier to audit.

### 1. Retail ELT with dbt and incremental loads

A retailer lands point-of-sale, e-commerce, and inventory feeds in S3, loads them with `COPY INTO`, and transforms them with dbt models. Daily volume is roughly 400 million order lines and 2 TB of raw files, which makes file sizing, incremental models, and pruning the difference between a pipeline that finishes before the business day and one that does not.

Loading runs on a dedicated ELT warehouse, transformation on a separate warehouse, and BI on a third, so a heavy backfill cannot slow a dashboard. Raw tables are loaded in date order, the `MERGE` consumer is idempotent so a retried batch is safe, and the fact table is clustered on `(order_date, customer_id)` so month-over-month reporting stays cheap. Query tags attribute every dbt model's credits back to its owner.

```sql
COPY INTO raw.orders.orders FROM @raw.orders.landing/order_date=2026-10-04/
  FILE_FORMAT = (TYPE = PARQUET) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  PATTERN = '.*[.]parquet' ON_ERROR = 'CONTINUE' PURGE = TRUE;

MERGE INTO analytics.fct_orders AS t
USING stg_orders AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET amount = s.amount, status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (order_id, customer_id, order_date, amount, status, updated_at)
VALUES (s.order_id, s.customer_id, s.order_date, s.amount, s.status, s.updated_at);

ALTER TABLE analytics.fct_orders CLUSTER BY (order_date, customer_id);
```

Why: loading and transformation use separate warehouses, the merge makes retries safe, and clustering on `(order_date, customer_id)` keeps month-over-month reporting scans small.

### 2. Fintech regulatory reporting with Time Travel

A licensed lender must reproduce balances and classifications exactly as they stood on a reporting date. Loads are append-only and immutable, and every reporting query pins a timestamp so the number in the report can be regenerated months later by anyone with the same privileges.

The reporting run clones the balance table at the reporting timestamp, computes the report against the immutable clone, and stores the `query_id` alongside the output for audit. Secure views and masking policies keep personally identifiable fields out of the analyst role, and Time Travel retention is set deliberately on the tables that back regulatory numbers rather than left at the default.

```sql
CREATE TABLE analytics.regulatory_snapshot CLONE analytics.fct_balances
  AT (TIMESTAMP => '2026-09-30 23:59:59'::TIMESTAMP);

SELECT customer_id,
       SUM(CASE WHEN balance >= 10000 THEN balance ELSE 0 END) AS reportable_balance
FROM analytics.regulatory_snapshot
GROUP BY customer_id;

ALTER TABLE analytics.fct_balances SET DATA_RETENTION_TIME_IN_DAYS = 90;
```

Why: Time Travel plus clones make point-in-time reporting a query rather than a restore exercise, and secure views with masking policies keep PII out of the analyst role.

### 3. Multi-tenant SaaS analytics with row access policies

A SaaS vendor serves every customer from one copy of the data but must guarantee tenant isolation. A mapping table links each role to the tenant ids it may see, and a row access policy on the fact tables enforces the filter for every consumer, including BI tools that never add a tenant predicate of their own.

Because isolation is enforced by the engine rather than by each application, one warehouse and one table serve everyone, and onboarding a tenant is a row in the mapping table rather than a new pipeline. The tradeoff is that policy logic runs per query, so the mapping table is kept small and indexed by role, and support access is granted through a narrowly scoped role rather than through `ACCOUNTADMIN`.

```sql
CREATE OR REPLACE ROW ACCESS POLICY security.tenant_policy
  AS (tenant_id STRING) RETURNS BOOLEAN ->
    CURRENT_ROLE() IN ('ACCOUNTADMIN', 'TENANT_ADMIN')
    OR EXISTS (
      SELECT 1 FROM security.tenant_map m
      WHERE m.role_name = CURRENT_ROLE() AND m.tenant_id = tenant_id
    );

ALTER TABLE analytics.fct_usage ADD ROW ACCESS POLICY security.tenant_policy ON (tenant_id);
```

Why: isolation is enforced by the engine rather than by each dashboard, so a misconfigured report cannot leak another tenant's data and one copy serves everyone.

### 4. Streaming CDC with Snowpipe Streaming and streams

A payments platform emits change events to Kafka, a connector writes them through Snowpipe Streaming into a raw table, a stream captures the delta, and a task merges it into the serving table every few minutes. End-to-end latency stays under ten minutes without any file landing step, which matters because payment status is customer-visible.

The stream makes each task incremental, so the merge processes only what changed, and the gating condition `SYSTEM$STREAM_HAS_DATA` means the task does nothing when there is no data. The merge is written to be safe to retry because delivery is at-least-once, and task history is monitored for failures and for long-running merges that would indicate the delta has outgrown the schedule.

```sql
CREATE OR REPLACE TASK analytics.apply_payments
  WAREHOUSE = transform_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw.payments.payments_stream')
AS
  MERGE INTO analytics.fct_payments AS t
  USING (
    SELECT payment_id, status, amount, updated_at, METADATA$ACTION AS action
    FROM raw.payments.payments_stream
    QUALIFY ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY updated_at DESC) = 1
  ) AS s
  ON t.payment_id = s.payment_id
  WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
  WHEN MATCHED THEN UPDATE SET status = s.status, amount = s.amount
  WHEN NOT MATCHED AND s.action = 'INSERT'
    THEN INSERT (payment_id, status, amount) VALUES (s.payment_id, s.status, s.amount);

ALTER TASK analytics.apply_payments RESUME;
```

Why: Snowpipe Streaming removes file-landing latency, the stream makes each task incremental, and the merge makes a retried task safe.

### 5. Cost-optimization program with resource monitors

A platform team runs dozens of warehouses and must cut spend without hurting SLAs. The program starts with attribution: credits per warehouse, per user, and per query tag, so the conversation moves from opinions to numbers. Then it right-sizes the biggest consumers, shortens auto-suspend on batch warehouses, and removes full refreshes that a `MERGE` or a dynamic table would do incrementally.

Guardrails come next. Resource monitors notify at a fraction of the quota and suspend at the limit, an account-level monitor catches warehouses without their own, and each warehouse has a named owner who sees its weekly credits. Changes are rolled out one at a time and re-measured, because credits per query is the metric that proves an optimization worked.

```sql
ALTER SESSION SET QUERY_TAG = 'team:marketing_pipeline';

SELECT warehouse_name,
       SUM(credits_used) AS credits,
       ROUND(SUM(credits_used) * 3.00, 2) AS approx_usd
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;

CREATE OR REPLACE RESOURCE MONITOR marketing_monitor WITH CREDIT_QUOTA = 500
  FREQUENCY = MONTHLY
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
```

Why: credits are attributable per warehouse, user, and query tag, so the program is measured rather than argued about; typical wins are shorter auto-suspend, right-sizing, fewer full refreshes, and separated workload compute.

### 6. Lakehouse with Iceberg tables shared across engines

A media company keeps one copy of its event data as Iceberg on S3. Snowflake writes the curated tables, Spark reads and backfills them, and Trino serves ad-hoc queries against the same Parquet files and metadata. No engine is locked out and the organization stops paying to copy the same data into a warehouse-native format for every consumer.

The design decision is write ownership. Snowflake-managed Iceberg tables are written through Snowflake and support native features such as `MERGE` and Time Travel, while catalog-linked tables are treated as read-mostly when an external catalog owns the writes. Storage lifecycle, file compaction, and catalog choice become explicit responsibilities rather than hidden ones.

```sql
CREATE ICEBERG TABLE analytics.events (
  event_id STRING, event_date DATE, customer_id NUMBER, payload VARIANT)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'iceberg_vol'
  BASE_LOCATION = 'analytics/events';

INSERT INTO analytics.events
SELECT event_id, event_date, customer_id, payload
FROM raw.events_staging
WHERE event_date = '2026-10-04';
```

Why: the table is open, so no engine is locked out and the organization stops paying to copy the same data into a warehouse-native format for every consumer, at the cost of making write ownership, maintenance, and catalog choice explicit.